# Biohub Cell Tracking — Batch Submission (Pipeline v1.8)

Self-contained notebook generated from `src/biohub/`.  
Regenerate locally: `python scripts/build_notebook.py`

This notebook exports the current classical production pipeline. No official
score or world-class ranking is claimed by this source file. See
`EXPERIMENTS.md` for dated, reproducible evaluation evidence.

**Steps**
1. Add the Biohub dataset to this notebook.
2. Run all cells and inspect the schema and graph validation result.
3. Submit `submission.csv` only after checking the generated artifact.

| Setting | Default |
|---------|---------|
| Output | `submission.csv` |
| Preset | v4 competition baseline (v1.5 features off) |
| Tuning | Optional train grid search + density calibration |
| V8 rule | Change **one** parameter per submission |


In [ ]:
import gc
import json
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

try:
    from skimage.feature import peak_local_max
    from skimage.filters import threshold_otsu
except ImportError:
    peak_local_max = None
    threshold_otsu = None

try:
    import blosc2
except ImportError:
    blosc2 = None

try:
    import zarr
except ImportError:
    zarr = None

warnings.filterwarnings("ignore")
print("Imports OK")


## Configuration


In [ ]:
"""Pipeline configuration and physical scale constants."""

from __future__ import annotations

from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Tuple

import numpy as np

# Physical voxel spacing (Z, Y, X) in micrometers.
SCALE: Tuple[float, float, float] = (1.625, 0.40625, 0.40625)
MATCH_GATE_UM: float = 7.0
PIPELINE_VERSION: str = "1.8"


@dataclass
class Config:
    """Pipeline hyperparameters for detection, tracking, and lineage reconstruction."""

    data_root: Path | None = None
    train_dir: Path | None = None
    test_dir: Path | None = None
    output_dir: Path = field(default_factory=lambda: Path("outputs"))
    output_path: Path = field(default_factory=lambda: Path("outputs/lineage_graph.csv"))
    voxel_size_zyx_um: Tuple[float, float, float] = SCALE
    frame_interval_s: float = 1.0

  # peaks = Gaussian-smoothed peaks (V7 @ 0.659); peaks_dog = DoG band-pass (V10)
    detector_backend: str = "peaks"
    learned_detector_path: Path = field(
        default_factory=lambda: Path("models/temporal_unet/edge_predictor_best.pth")
    )
    learned_detector_threshold: float = 0.50
    learned_detector_pool_um: float = 5.0
    learned_detector_downsample: Tuple[int, int, int] = (1, 4, 4)
    learned_detector_device: str = "auto"

    # Detection
    xy_ds: int = 4
    smooth_sigma: float = 1.0
    # DoG band-pass preprocessing (opt-in â€” V10 competition experiment)
    use_dog_bandpass: bool = False
    dog_sigma_small_um: float = 1.0
    dog_sigma_large_um: float = 2.8
    dog_clip_negative: bool = True
    dog_post_smooth_sigma: float = 0.0
    min_peak_dist: int = 3
    thresh_rel: float = 0.30
    thresh_hi_percentile: float = 99.8
    min_rel_contrast: float = 0.08
    use_dense_cluster_pass: bool = True
    dense_min_peak_dist: int = 2
    dense_thresh_rel_delta: float = -0.04
    use_adaptive_frame_threshold: bool = True
    adaptive_overcount_penalty: float = 0.04
    min_z_hard: int = 4

    refine_radius_z: int = 2
    refine_radius_yx: int = 5
    nms_radius_um: float = 2.65
    nms_dense_radius_um: float = 2.0
    border_z: int = 3
    border_yx: int = 2
    border_keep_quantile: float = 0.80

    max_frame_count_mult: float = 1.70
    max_frame_count_add: int = 45
    max_nodes_per_frame: int = 20_000

    # Linking
    max_link_dist_um: float = 11.0
    linker_backend: str = "classical"
    learned_linker_path: Path = field(
        default_factory=lambda: Path("models/full_annotation_linker/final_all_annotations.pt")
    )
    learned_linker_gate_um: float = 20.0
    image_linker_path: Path = field(
        default_factory=lambda: Path("models/full_image_linker/image_final_all_199.pt")
    )
    use_rich_linking: bool = True
    tracking_backend: str = "framewise"
    global_max_frame_gap: int = 2
    global_gap_penalty: float = 0.35
    global_unmatched_cost: float = 0.75
    motion_lambda: float = 0.20
    intensity_lambda: float = 0.15
    neighborhood_lambda: float = 0.10
    neighborhood_k: int = 4
    neighborhood_radius_um: float = 15.0

    # Divisions
    detect_divisions: bool = True
    div_parent_dist_um: float = 12.0
    div_sister_dist_um: float = 7.5
    div_min_count_gain: int = 0
    div_require_continued: bool = False
    div_use_midpoint_gate: bool = True
    div_midpoint_dist_um: float = 9.0
    joint_hypothesis_unmatched_cost: float = 0.75
    joint_division_geometry_weight: float = 0.50

    prune_isolated_nodes: bool = True
    # v1.5 experiments (disabled by default â€” regressed leaderboard v5 vs v4)
    prune_soft_neighbors: bool = False
    prune_neighbor_dist_um: float = 9.0
    gap_close_enabled: bool = False
    gap_close_dist_um: float = 15.0
    div_symmetry_weight: float = 0.0

    # Use competition preset for batch submission (matches v4 @ 0.659)
    use_competition_preset: bool = True

    # Tuning
    run_hyperparameter_search: bool = False
    run_density_calibration: bool = True
    hyperparam_sample_limit: int = 3
    hyperparam_frames: int = 4
    hyperparam_results_path: Path = field(default_factory=lambda: Path("results/hyperparameter_search.csv"))

    preview_max_frames: int = 20
    eda_sample_limit: int = 4
    calibration_frames: int = 5
    random_state: int = 42
    use_detection_cache: bool = True
    detection_cache_dir: Path = field(default_factory=lambda: Path("results/detection_cache"))

    def __post_init__(self) -> None:
        if len(self.voxel_size_zyx_um) != 3 or any(v <= 0 for v in self.voxel_size_zyx_um):
            raise ValueError("voxel_size_zyx_um must contain three positive values")
        if self.frame_interval_s <= 0:
            raise ValueError("frame_interval_s must be positive")
        for name in (
            "thresh_rel", "min_rel_contrast", "border_keep_quantile",
            "learned_detector_threshold",
        ):
            value = float(getattr(self, name))
            if not 0 <= value <= 1:
                raise ValueError(f"{name} must be in [0, 1]")
        for name in (
            "dog_sigma_small_um", "dog_sigma_large_um", "nms_radius_um",
            "max_link_dist_um", "learned_linker_gate_um", "div_parent_dist_um",
            "div_sister_dist_um", "div_midpoint_dist_um",
        ):
            if float(getattr(self, name)) <= 0:
                raise ValueError(f"{name} must be positive")
        if self.dog_sigma_large_um <= self.dog_sigma_small_um:
            raise ValueError("dog_sigma_large_um must exceed dog_sigma_small_um")
        if self.detector_backend not in {"peaks", "peaks_dog", "learned"}:
            raise ValueError("detector_backend must be 'peaks', 'peaks_dog', or 'learned'")
        if len(self.learned_detector_downsample) != 3 or any(
            not isinstance(v, int) or isinstance(v, bool) or v < 1
            for v in self.learned_detector_downsample
        ):
            raise ValueError("learned_detector_downsample must contain three positive integers")
        if self.learned_detector_pool_um <= 0:
            raise ValueError("learned_detector_pool_um must be positive")
        if self.learned_detector_device not in {"auto", "cpu", "cuda"}:
            raise ValueError("learned_detector_device must be 'auto', 'cpu', or 'cuda'")
        if self.tracking_backend not in {"framewise", "global"}:
            raise ValueError("tracking_backend must be 'framewise' or 'global'")
        if self.global_max_frame_gap < 1:
            raise ValueError("global_max_frame_gap must be at least 1")
        if self.global_gap_penalty < 0 or self.global_unmatched_cost <= 0:
            raise ValueError("global tracking costs must be non-negative and unmatched cost positive")
        if self.joint_hypothesis_unmatched_cost <= 0 or self.joint_division_geometry_weight < 0:
            raise ValueError("joint hypothesis unmatched cost must be positive and geometry weight non-negative")

    @property
    def scale_array(self) -> np.ndarray:
        return np.array(self.voxel_size_zyx_um, dtype=np.float64)

    def resolve_paths(self, project_root: Path | None = None) -> None:
        """Resolve default data directories relative to the project root."""
        root = project_root or Path.cwd()

        # Hosted notebook runtimes (e.g. cloud notebook with read-only input mount).
        hosted_roots = [
            Path("/kaggle/input/competitions/biohub-cell-tracking-during-development"),
            Path("/kaggle/input/biohub-cell-tracking-during-development"),
        ]
        for comp_root in hosted_roots:
            if (comp_root / "test").is_dir():
                self.data_root = comp_root
                self.test_dir = comp_root / "test"
                self.train_dir = comp_root / "train" if (comp_root / "train").is_dir() else None
                if Path("/kaggle/working").is_dir():
                    self.output_path = Path("/kaggle/working/submission.csv")
                self.output_dir.mkdir(parents=True, exist_ok=True)
                return

        candidates = [
            root / "data",
            root / "data" / "train",
            root.parent / "data",
        ]
        if self.data_root is None:
            self.data_root = next((p for p in candidates if p.is_dir()), None)
        if self.data_root is not None:
            if self.train_dir is None:
                train = self.data_root / "train" if (self.data_root / "train").is_dir() else self.data_root
                self.train_dir = train if train.is_dir() else None
            if self.test_dir is None:
                test = self.data_root / "test"
                self.test_dir = test if test.is_dir() else None
        self.output_dir.mkdir(parents=True, exist_ok=True)
        if self.output_path.parent == Path("."):
            self.output_path = self.output_dir / self.output_path.name

    def copy_with(self, **kwargs) -> "Config":
        return replace(self, **kwargs)

    def competition_v4_preset(self) -> "Config":
        """
        Leaderboard v4 baseline (public score 0.659).

        Disables v1.5 features that correlated with the v5 regression (0.648).
        """
        return self.copy_with(
            gap_close_enabled=False,
            prune_soft_neighbors=False,
            div_symmetry_weight=0.0,
            use_adaptive_frame_threshold=False,
            use_dog_bandpass=False,
            detector_backend="peaks",
            linker_backend="classical",
        )

    def competition_v10_dog_preset(self) -> "Config":
        """
        V10 experiment: v4 tracking preset + DoG band-pass detection front-end.

        Keeps validated linking/division settings; only changes preprocessing.
        """
        return self.competition_v4_preset().copy_with(
            detector_backend="peaks_dog",
            use_dog_bandpass=True,
            dog_sigma_small_um=1.0,
            dog_sigma_large_um=2.8,
            dog_clip_negative=True,
            dog_post_smooth_sigma=0.0,
            use_competition_preset=False,
        )


## Data I/O


In [ ]:
"""Zarr volume and GEFF graph loading."""

from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np

logger = logging.getLogger(__name__)

try:
    import blosc2
except ImportError:
    blosc2 = None  # type: ignore

try:
    import zarr
except ImportError:
    zarr = None  # type: ignore


def list_datasets(split_dir: Path) -> List[str]:
    """Return sorted dataset names (without .zarr suffix)."""
    if not split_dir or not split_dir.is_dir():
        return []
    return sorted(p.name[:-5] for p in split_dir.iterdir() if p.name.endswith(".zarr"))


def read_zarr_meta(zarr_path: Path) -> Tuple[Tuple[int, ...], np.dtype]:
    """Read shape and dtype from zarr metadata."""
    meta_path = zarr_path / "0" / "zarr.json"
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing metadata: {meta_path}")
    meta = json.loads(meta_path.read_text())
    return tuple(meta["shape"]), np.dtype(meta["data_type"])


def read_physical_metadata(zarr_path: Path) -> Tuple[Optional[Tuple[float, float, float]], Optional[float]]:
    """Read OME-Zarr spatial voxel size and frame interval when declared."""
    meta_path = zarr_path / "zarr.json"
    if not meta_path.exists():
        return None, None
    try:
        attrs = json.loads(meta_path.read_text()).get("attributes", {})
        multiscale = attrs.get("multiscales", [])[0]
        axes = multiscale.get("axes", [])
        dataset = multiscale.get("datasets", [])[0]
        transform = next(
            item for item in dataset.get("coordinateTransformations", [])
            if item.get("type") == "scale"
        )
        values = [float(v) for v in transform["scale"]]
        names = [str(axis.get("name", "")).lower() for axis in axes]
        mapped = dict(zip(names, values))
        voxel = tuple(mapped[k] for k in ("z", "y", "x"))
        frame_interval = mapped.get("t")
        return voxel, float(frame_interval) if frame_interval is not None else None
    except (IndexError, KeyError, StopIteration, TypeError, ValueError):
        return None, None


def load_volume(
    zarr_path: Path,
    t: int,
    shape: Tuple[int, ...],
    dtype: np.dtype,
) -> np.ndarray:
    """Load one timepoint as (Z, Y, X)."""
    chunk = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"
    if blosc2 is not None and chunk.exists():
        raw = blosc2.decompress(chunk.read_bytes())
        return np.frombuffer(raw, dtype=dtype).reshape(shape[1:])
    if zarr is not None:
        arr = zarr.open_array(str(zarr_path / "0"), mode="r")
        return np.asarray(arr[t], dtype=dtype)
    raise ImportError("blosc2 or zarr required to read volumes")


def read_estimated_nodes(geff_path: Path) -> float | None:
    """Read estimated_number_of_nodes from GEFF metadata if present."""
    meta_path = geff_path / "zarr.json"
    if not meta_path.exists():
        return None
    try:
        meta = json.loads(meta_path.read_text())
    except json.JSONDecodeError:
        return None

    def _find(obj):
        if isinstance(obj, dict):
            if "estimated_number_of_nodes" in obj:
                return obj["estimated_number_of_nodes"]
            for v in obj.values():
                found = _find(v)
                if found is not None:
                    return found
        elif isinstance(obj, list):
            for item in obj:
                found = _find(item)
                if found is not None:
                    return found
        return None

    val = _find(meta)
    return float(val) if val is not None else None


def find_geff_path(split_dir: Path, name: str) -> Optional[Path]:
    """Locate a GEFF graph for ``name`` across common competition layouts."""
    candidates = [
        split_dir / f"{name}.geff",
        split_dir / "geff" / f"{name}.geff",
        split_dir / "graphs" / f"{name}.geff",
        split_dir / "labels" / f"{name}.geff",
        split_dir.parent / "geff" / f"{name}.geff",
        split_dir.parent / "graphs" / f"{name}.geff",
        split_dir.parent / "labels" / f"{name}.geff",
    ]
    return next((p for p in candidates if p.exists()), None)


def _read_single_chunk_zarr3(array_path: Path) -> np.ndarray:
    """Read the single-chunk Zarr v3 arrays used by current GEFF labels.

    This compatibility path is needed on Python 3.10, where stable zarr 3
    releases are unavailable. It fails loudly if a future GEFF array uses
    multiple chunks or an unsupported codec.
    """
    metadata = json.loads((array_path / "zarr.json").read_text())
    shape = tuple(int(v) for v in metadata["shape"])
    chunk_shape = tuple(int(v) for v in metadata["chunk_grid"]["configuration"]["chunk_shape"])
    if chunk_shape != shape:
        raise ValueError(f"Expected a single chunk at {array_path}; shape={shape}, chunk={chunk_shape}")
    key = array_path / "c"
    for _ in shape:
        key /= "0"
    payload = key.read_bytes()
    endian = "="
    for codec in reversed(metadata.get("codecs", [])):
        name = codec["name"]
        if name == "zstd":
            from numcodecs import Zstd

            payload = Zstd().decode(payload)
        elif name == "bytes":
            endian = "<" if codec.get("configuration", {}).get("endian") == "little" else ">"
        else:
            raise ValueError(f"Unsupported Zarr v3 codec {name!r} at {array_path}")
    dtype = np.dtype(metadata["data_type"]).newbyteorder(endian)
    expected = int(np.prod(shape))
    values = np.frombuffer(payload, dtype=dtype, count=expected)
    if values.size != expected:
        raise ValueError(f"Decoded {values.size} values at {array_path}; expected {expected}")
    return values.reshape(shape).copy()


def _load_geff_graph_zarr3_compat(geff_path: Path):
    import pandas as pd

    nodes = pd.DataFrame(
        {
            "node_id": _read_single_chunk_zarr3(geff_path / "nodes" / "ids"),
            "t": _read_single_chunk_zarr3(geff_path / "nodes" / "props" / "t" / "values"),
            "z": _read_single_chunk_zarr3(geff_path / "nodes" / "props" / "z" / "values"),
            "y": _read_single_chunk_zarr3(geff_path / "nodes" / "props" / "y" / "values"),
            "x": _read_single_chunk_zarr3(geff_path / "nodes" / "props" / "x" / "values"),
        }
    )
    edge_ids = _read_single_chunk_zarr3(geff_path / "edges" / "ids")
    edges = pd.DataFrame({"source_id": edge_ids[:, 0], "target_id": edge_ids[:, 1]})
    return nodes, edges


def load_geff_graph(geff_path: Path):
    """Load node and edge tables from a GEFF directory."""
    if not geff_path.exists():
        raise FileNotFoundError(f"GEFF directory not found: {geff_path}")
    try:
        import pandas as pd

        if zarr is None:
            raise ImportError('zarr unavailable; attempting explicit GEFF compatibility reader')
        root = zarr.open_group(str(geff_path), mode="r")
        nodes_grp = root["nodes"]
        edges_grp = root["edges"]
        node_ids = np.asarray(nodes_grp["ids"])
        props = nodes_grp["props"]
        nodes = pd.DataFrame(
            {
                "node_id": node_ids,
                "t": np.asarray(props["t"]["values"]),
                "z": np.asarray(props["z"]["values"]),
                "y": np.asarray(props["y"]["values"]),
                "x": np.asarray(props["x"]["values"]),
            }
        )
        edge_ids = np.asarray(edges_grp["ids"])
        edges = pd.DataFrame({"source_id": edge_ids[:, 0], "target_id": edge_ids[:, 1]})
        return nodes, edges
    except Exception as exc:
        compat_error: Exception | None = None
        try:
            nodes, edges = _load_geff_graph_zarr3_compat(geff_path)
            logger.warning(
                "Stable GEFF reader failed for %s (%s); used explicit Zarr v3 compatibility reader",
                geff_path,
                exc,
            )
            return nodes, edges
        except Exception as compat_exc:
            compat_error = compat_exc
        contents = sorted(
            str(p.relative_to(geff_path))
            for p in geff_path.rglob("*")
            if p.is_file()
        )
        message = (
            f"Failed to load GEFF graph at {geff_path}: primary={exc}; "
            f"compatibility={compat_error}. "
            f"Found {len(contents)} files; first entries: {contents[:20]}"
        )
        logger.exception(message)
        raise RuntimeError(message) from exc


## Detection


In [ ]:
"""Anisotropy-aware cell centroid detection (v1.1)."""

from __future__ import annotations

from typing import Optional, Tuple

import numpy as np
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import cKDTree


try:
    from skimage.feature import peak_local_max
    from skimage.filters import threshold_otsu
except ImportError:
    peak_local_max = None
    threshold_otsu = None


def block_mean_xy(vol: np.ndarray, factor: int) -> np.ndarray:
    """Average-pool XY while preserving Z. Makes the grid ~isotropic in microns."""
    z, y, x = vol.shape
    y2, x2 = (y // factor) * factor, (x // factor) * factor
    arr = vol[:, :y2, :x2].astype(np.float32, copy=False)
    return arr.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4))


def _anisotropic_sigma_voxels(sigma_um: float, scale_zyx: np.ndarray, xy_ds: int) -> Tuple[float, float, float]:
    """Convert a physical Gaussian sigma (Âµm) to voxel sigmas on the XY-downsampled grid."""
    eff = scale_zyx.astype(np.float64).copy()
    eff[1] *= xy_ds
    eff[2] *= xy_ds
    return tuple(float(sigma_um / s) for s in eff)


def dog_bandpass(ds: np.ndarray, cfg: Config) -> np.ndarray:
    """
    Difference-of-Gaussians band-pass on the downsampled volume.

    Enhances blob-like bright structures (cell-sized) while suppressing slowly
    varying background. Used by high-scoring public notebooks (~0.85 LB).
    """
    s_small = _anisotropic_sigma_voxels(cfg.dog_sigma_small_um, cfg.scale_array, cfg.xy_ds)
    s_large = _anisotropic_sigma_voxels(cfg.dog_sigma_large_um, cfg.scale_array, cfg.xy_ds)
    arr = ds.astype(np.float32, copy=False)
    g_small = gaussian_filter(arr, sigma=s_small, mode="nearest")
    g_large = gaussian_filter(arr, sigma=s_large, mode="nearest")
    dog = g_small - g_large
    if cfg.dog_clip_negative:
        dog = np.maximum(dog, 0.0)
    if cfg.dog_post_smooth_sigma > 0:
        dog = gaussian_filter(dog, sigma=cfg.dog_post_smooth_sigma, mode="nearest")
    return dog.astype(np.float32, copy=False)


def preprocess_detection_volume(ds: np.ndarray, cfg: Config) -> np.ndarray:
    """Return the smoothed or band-pass filtered volume used for peak finding."""
    if cfg.use_dog_bandpass or cfg.detector_backend == "peaks_dog":
        return dog_bandpass(ds, cfg)
    return gaussian_filter(ds, sigma=cfg.smooth_sigma, mode="nearest")


def robust_threshold(
    sm: np.ndarray,
    cfg: Config,
    thresh_rel: Optional[float] = None,
) -> Tuple[float, float, float]:
    """Otsu with a relative-rise floor; optional per-pass threshold override."""
    alpha = cfg.thresh_rel if thresh_rel is None else float(thresh_rel)
    bg = float(np.median(sm))
    hi = float(np.percentile(sm, cfg.thresh_hi_percentile))
    dyn = max(hi - bg, 1e-6)
    rel_thr = bg + alpha * dyn
    try:
        otsu = float(threshold_otsu(sm)) if threshold_otsu else float(np.percentile(sm, 96.0))
    except Exception:
        otsu = float(np.percentile(sm, 96.0))
    return max(otsu, rel_thr), bg, dyn


def adaptive_thresh_rel(cfg: Config, prev_count: Optional[int], n_detected: int) -> float:
    """Raise threshold when the current frame exceeds the adaptive count cap."""
    rel = cfg.thresh_rel
    if not cfg.use_adaptive_frame_threshold or prev_count is None or prev_count < 8:
        return rel
    cap = int(prev_count * cfg.max_frame_count_mult + cfg.max_frame_count_add)
    if n_detected > cap:
        rel = min(rel + cfg.adaptive_overcount_penalty, 0.42)
    return rel


def _fallback_peaks(sm: np.ndarray, threshold_abs: float, min_distance: int) -> np.ndarray:
    size = 2 * min_distance + 1
    mx = maximum_filter(sm, size=(size, size, size), mode="nearest")
    mask = (sm >= mx) & (sm > threshold_abs)
    coords = np.argwhere(mask)
    if coords.size == 0:
        return coords.astype(np.int32)
    vals = sm[coords[:, 0], coords[:, 1], coords[:, 2]]
    return coords[np.argsort(-vals)].astype(np.int32)


def _find_peaks(sm: np.ndarray, threshold_abs: float, min_distance: int) -> np.ndarray:
    if peak_local_max is not None:
        return peak_local_max(
            sm,
            min_distance=min_distance,
            threshold_abs=threshold_abs,
            exclude_border=False,
        ).astype(np.int32)
    return _fallback_peaks(sm, threshold_abs, min_distance)


def refine_centroid(
    vol: np.ndarray,
    approx_zyx: np.ndarray,
    cfg: Config,
) -> Tuple[np.ndarray, float]:
    """Intensity-weighted centroid in a local neighborhood."""
    zc, yc, xc = [int(round(v)) for v in approx_zyx]
    z0 = max(0, zc - cfg.refine_radius_z)
    z1 = min(vol.shape[0], zc + cfg.refine_radius_z + 1)
    y0 = max(0, yc - cfg.refine_radius_yx)
    y1 = min(vol.shape[1], yc + cfg.refine_radius_yx + 1)
    x0 = max(0, xc - cfg.refine_radius_yx)
    x1 = min(vol.shape[2], xc + cfg.refine_radius_yx + 1)
    crop = vol[z0:z1, y0:y1, x0:x1].astype(np.float32, copy=False)
    if crop.size == 0:
        return approx_zyx.astype(np.float64), 0.0

    bg = float(np.percentile(crop, 20.0))
    weights = np.clip(crop - bg, 0.0, None)
    total = float(weights.sum())
    if total <= 1e-6:
        loc = np.unravel_index(int(np.argmax(crop)), crop.shape)
        return np.array([z0 + loc[0], y0 + loc[1], x0 + loc[2]], dtype=np.float64), float(crop[loc])

    zz, yy, xx = np.indices(crop.shape)
    refined = np.array(
        [
            z0 + float((zz * weights).sum() / total),
            y0 + float((yy * weights).sum() / total),
            x0 + float((xx * weights).sum() / total),
        ],
        dtype=np.float64,
    )
    return refined, float(weights.max())


def sample_intensity(vol: np.ndarray, coords: np.ndarray) -> np.ndarray:
    """Max intensity in a 3x3x3 neighborhood at each centroid."""
    if len(coords) == 0:
        return np.empty((0,), dtype=np.float32)
    out = np.empty(len(coords), dtype=np.float32)
    z_dim, y_dim, x_dim = vol.shape
    for i, (z, y, x) in enumerate(coords):
        zc, yc, xc = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, zc - 1), min(z_dim, zc + 2)
        y0, y1 = max(0, yc - 1), min(y_dim, yc + 2)
        x0, x1 = max(0, xc - 1), min(x_dim, xc + 2)
        out[i] = float(vol[z0:z1, y0:y1, x0:x1].max())
    return out


def physical_nms(
    coords_vox: np.ndarray,
    scores: np.ndarray,
    radius_um: float,
    scale: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    if len(coords_vox) <= 1:
        return coords_vox, scores
    pts = coords_vox.astype(np.float64) * scale[None, :]
    order = np.argsort(-scores)
    tree = cKDTree(pts)
    suppressed = np.zeros(len(coords_vox), dtype=bool)
    keep = []
    for i in order:
        if suppressed[i]:
            continue
        keep.append(i)
        for j in tree.query_ball_point(pts[i], r=radius_um):
            suppressed[j] = True
    keep = np.array(keep, dtype=np.int64)
    return coords_vox[keep], scores[keep]


def _map_ds_to_full(coords_ds: np.ndarray, xy_ds: int) -> np.ndarray:
    approx = coords_ds.astype(np.float64)
    approx[:, 1] = approx[:, 1] * xy_ds + (xy_ds - 1) / 2.0
    approx[:, 2] = approx[:, 2] * xy_ds + (xy_ds - 1) / 2.0
    return approx


def _merge_peak_sets(
    primary: np.ndarray,
    secondary: np.ndarray,
    sm: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    if len(primary) == 0 and len(secondary) == 0:
        return primary, np.empty((0,), dtype=np.float32)
    if len(primary) == 0:
        coords = secondary
    elif len(secondary) == 0:
        coords = primary
    else:
        coords = np.vstack([primary, secondary])
        coords = np.unique(coords, axis=0)
    scores = sm[coords[:, 0], coords[:, 1], coords[:, 2]].astype(np.float32)
    return coords, scores


def _extract_from_sm(
    sm: np.ndarray,
    vol: np.ndarray,
    cfg: Config,
    thresh_rel: float,
    prev_count: Optional[int],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Run peak finding through NMS on a pre-smoothed downsampled volume."""
    z_dim, y_dim, x_dim = vol.shape
    threshold_abs, bg, dyn = robust_threshold(sm, cfg, thresh_rel=thresh_rel)

    coords_primary = _find_peaks(sm, threshold_abs, cfg.min_peak_dist)
    coords_dense = np.empty((0, 3), dtype=np.int32)
    if cfg.use_dense_cluster_pass:
        dense_rel = max(0.08, thresh_rel + cfg.dense_thresh_rel_delta)
        dense_thr, _, _ = robust_threshold(sm, cfg, thresh_rel=dense_rel)
        coords_dense = _find_peaks(sm, dense_thr, cfg.dense_min_peak_dist)

    coords_ds, peak_scores = _merge_peak_sets(coords_primary, coords_dense, sm)
    if coords_ds.size == 0:
        flat = np.argpartition(sm.ravel(), -3)[-3:]
        coords_ds = np.array(np.unravel_index(flat, sm.shape)).T.astype(np.int32)
        peak_scores = sm[coords_ds[:, 0], coords_ds[:, 1], coords_ds[:, 2]].astype(np.float32)

    rel_contrast = (peak_scores - bg) / max(dyn, 1e-6)
    keep = rel_contrast >= cfg.min_rel_contrast
    coords_ds, peak_scores = coords_ds[keep], peak_scores[keep]
    if len(coords_ds) == 0:
        return (
            np.empty((0, 3), dtype=np.int32),
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
        )

    approx = _map_ds_to_full(coords_ds, cfg.xy_ds)
    refined, refined_scores = [], []
    for a, s in zip(approx, peak_scores):
        r, rs = refine_centroid(vol, a, cfg)
        refined.append(r)
        refined_scores.append(max(float(s), rs))
    coords = np.vstack(refined)
    scores = np.array(refined_scores, dtype=np.float32)

    if cfg.min_z_hard > 0 and len(coords):
        z_floor_score = float(np.quantile(scores, 0.85)) if len(scores) > 8 else np.inf
        keep = (coords[:, 0] >= cfg.min_z_hard) | (scores >= z_floor_score)
        coords, scores = coords[keep], scores[keep]

    cz, cy, cx = coords[:, 0], coords[:, 1], coords[:, 2]
    border = (
        (cz <= cfg.border_z)
        | (cz >= z_dim - 1 - cfg.border_z)
        | (cy <= cfg.border_yx)
        | (cy >= y_dim - 1 - cfg.border_yx)
        | (cx <= cfg.border_yx)
        | (cx >= x_dim - 1 - cfg.border_yx)
    )
    floor = float(np.quantile(scores, cfg.border_keep_quantile)) if len(scores) > 8 else -np.inf
    keep = (~border) | (scores >= floor)
    coords, scores = coords[keep], scores[keep]

    coords, scores = physical_nms(coords, scores, cfg.nms_radius_um, cfg.scale_array)
    if cfg.use_dense_cluster_pass and len(coords) > 1:
        coords, scores = physical_nms(coords, scores, cfg.nms_dense_radius_um, cfg.scale_array)

    if prev_count is not None and prev_count >= 8:
        cap = int(prev_count * cfg.max_frame_count_mult + cfg.max_frame_count_add)
        if len(coords) > cap:
            order = np.argsort(-scores)[:cap]
            coords, scores = coords[order], scores[order]

    if len(coords) > cfg.max_nodes_per_frame:
        order = np.argsort(-scores)[: cfg.max_nodes_per_frame]
        coords, scores = coords[order], scores[order]

    coords = np.rint(coords).astype(np.int32)
    coords[:, 0] = np.clip(coords[:, 0], 0, z_dim - 1)
    coords[:, 1] = np.clip(coords[:, 1], 0, y_dim - 1)
    coords[:, 2] = np.clip(coords[:, 2], 0, x_dim - 1)
    intensities = sample_intensity(vol, coords)
    return coords, scores, intensities


def detect_cells(
    vol: np.ndarray,
    cfg: Config,
    prev_count: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Detect cell centroids in one 3D frame.

    Returns integer (z, y, x) coordinates, detector scores, and sampled intensities.
    """
    ds = block_mean_xy(vol, cfg.xy_ds)
    sm = preprocess_detection_volume(ds, cfg)

    thresh_rel = cfg.thresh_rel
    coords, scores, intensities = _extract_from_sm(sm, vol, cfg, thresh_rel, prev_count)

    # Adaptive retry: tighten threshold when this frame is over the adaptive cap.
    if cfg.use_adaptive_frame_threshold and prev_count is not None and prev_count >= 8:
        tighter = adaptive_thresh_rel(cfg, prev_count, len(coords))
        if tighter > thresh_rel + 1e-6:
            coords, scores, intensities = _extract_from_sm(sm, vol, cfg, tighter, prev_count)

    return coords, scores, intensities


## Tracking


In [ ]:
"""Frame-to-frame linking and division detection (v1.2 / v1.3)."""

from __future__ import annotations

from collections import Counter, defaultdict
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
from scipy.optimize import Bounds, LinearConstraint, linear_sum_assignment, milp
from scipy.spatial import cKDTree
from scipy.sparse import csc_matrix



def neighbor_signature(xyz_phys: np.ndarray, cfg: Config) -> np.ndarray:
    """K-nearest-neighbor offset vectors as a local context descriptor."""
    n = len(xyz_phys)
    if n == 0:
        return np.empty((0, cfg.neighborhood_k * 3), dtype=np.float64)
    k = min(cfg.neighborhood_k, max(n - 1, 0))
    if k == 0:
        return np.zeros((n, cfg.neighborhood_k * 3), dtype=np.float64)
    tree = cKDTree(xyz_phys)
    dists, idx = tree.query(xyz_phys, k=k + 1)
    sig = np.zeros((n, cfg.neighborhood_k * 3), dtype=np.float64)
    for i in range(n):
        nbrs = idx[i, 1:] if np.ndim(idx) > 1 else []
        for j, nb in enumerate(nbrs[: cfg.neighborhood_k]):
            if dists[i, j + 1] <= cfg.neighborhood_radius_um:
                sig[i, j * 3 : (j + 1) * 3] = xyz_phys[nb] - xyz_phys[i]
    return sig


def _intensity_cost(prev_i: np.ndarray, curr_j: np.ndarray) -> np.ndarray:
    if prev_i.size == 0 or curr_j.size == 0:
        return np.zeros((len(prev_i), len(curr_j)), dtype=np.float64)
    p = prev_i[:, None].astype(np.float64)
    c = curr_j[None, :].astype(np.float64)
    denom = np.maximum(np.maximum(p, c), 1.0)
    return np.abs(p - c) / denom


def _signature_cost(prev_sig: np.ndarray, curr_sig: np.ndarray) -> np.ndarray:
    if len(prev_sig) == 0 or len(curr_sig) == 0:
        return np.zeros((len(prev_sig), len(curr_sig)), dtype=np.float64)
    diff = prev_sig[:, None, :] - curr_sig[None, :, :]
    return np.sqrt((diff ** 2).sum(axis=2))


def build_link_cost(
    prev_phys: np.ndarray,
    curr_phys: np.ndarray,
    cfg: Config,
    prev_intensity: Optional[np.ndarray] = None,
    curr_intensity: Optional[np.ndarray] = None,
    prev_velocity: Optional[np.ndarray] = None,
    prev_sig: Optional[np.ndarray] = None,
    curr_sig: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """Physical distance matrix plus optional rich-linking terms."""
    dist = np.sqrt(((prev_phys[:, None, :] - curr_phys[None, :, :]) ** 2).sum(axis=2))
    cost = dist.copy()

    if cfg.use_rich_linking:
        if prev_velocity is not None and len(prev_velocity) == len(prev_phys):
            predicted = prev_phys[:, None, :] + prev_velocity[:, None, :]
            motion_resid = np.sqrt(((curr_phys[None, :, :] - predicted) ** 2).sum(axis=2))
            cost = cost + cfg.motion_lambda * motion_resid
        if prev_intensity is not None and curr_intensity is not None:
            cost = cost + cfg.intensity_lambda * _intensity_cost(prev_intensity, curr_intensity)
        if prev_sig is not None and curr_sig is not None:
            sig_cost = _signature_cost(prev_sig, curr_sig)
            med = float(np.median(sig_cost[sig_cost > 0])) if np.any(sig_cost > 0) else 1.0
            cost = cost + cfg.neighborhood_lambda * (sig_cost / max(med, 1e-6))

    big = 1e6
    cost = np.where(dist <= cfg.max_link_dist_um, cost, big)
    return cost, dist


def _candidate_components(
    prev_phys: np.ndarray,
    curr_phys: np.ndarray,
    gate_um: float,
) -> List[Tuple[List[int], List[int]]]:
    """Connected components of the sparse gated bipartite candidate graph."""
    tree = cKDTree(curr_phys)
    row_to_cols = [sorted(map(int, cols)) for cols in tree.query_ball_point(prev_phys, gate_um)]
    col_to_rows: Dict[int, List[int]] = defaultdict(list)
    for row, cols in enumerate(row_to_cols):
        for col in cols:
            col_to_rows[col].append(row)
    components: List[Tuple[List[int], List[int]]] = []
    seen_rows: set[int] = set()
    for start in range(len(prev_phys)):
        if start in seen_rows or not row_to_cols[start]:
            continue
        rows: set[int] = set()
        cols: set[int] = set()
        pending_rows = [start]
        while pending_rows:
            row = pending_rows.pop()
            if row in rows:
                continue
            rows.add(row)
            for col in row_to_cols[row]:
                if col not in cols:
                    cols.add(col)
                    pending_rows.extend(col_to_rows[col])
        seen_rows.update(rows)
        components.append((sorted(rows), sorted(cols)))
    return components


def _component_assignments(
    prev_phys: np.ndarray,
    curr_phys: np.ndarray,
    cfg: Config,
    prev_intensity: Optional[np.ndarray],
    curr_intensity: Optional[np.ndarray],
    prev_velocity: Optional[np.ndarray],
    prev_image_features: Optional[np.ndarray],
    curr_image_features: Optional[np.ndarray],
) -> List[Tuple[int, int]]:
    """Solve Hungarian assignment per sparse candidate component."""
    learned_backends = {"learned", "image_learned", "learned_joint", "image_learned_joint"}
    gate = cfg.learned_linker_gate_um if cfg.linker_backend in learned_backends else cfg.max_link_dist_um
    components = _candidate_components(prev_phys, curr_phys, gate)
    prev_sig = neighbor_signature(prev_phys, cfg) if cfg.use_rich_linking and cfg.linker_backend == "classical" else None
    curr_sig = neighbor_signature(curr_phys, cfg) if cfg.use_rich_linking and cfg.linker_backend == "classical" else None
    assignments: List[Tuple[int, int]] = []
    for rows, cols in components:
        row_ix = np.asarray(rows, dtype=int)
        col_ix = np.asarray(cols, dtype=int)
        if cfg.linker_backend in learned_backends:

            cost, _ = build_learned_link_cost(
                prev_phys[row_ix],
                curr_phys[col_ix],
                cfg.image_linker_path if cfg.linker_backend in {"image_learned", "image_learned_joint"} else cfg.learned_linker_path,
                gate_um=gate,
                prev_image_features=None if prev_image_features is None else prev_image_features[row_ix],
                curr_image_features=None if curr_image_features is None else curr_image_features[col_ix],
            )
        elif cfg.linker_backend in {"classical", "classical_joint"}:
            cost, _ = build_link_cost(
                prev_phys[row_ix], curr_phys[col_ix], cfg,
                prev_intensity=None if prev_intensity is None else prev_intensity[row_ix],
                curr_intensity=None if curr_intensity is None else curr_intensity[col_ix],
                prev_velocity=(
                    None
                    if prev_velocity is None or len(prev_velocity) != len(prev_phys)
                    else prev_velocity[row_ix]
                ),
                prev_sig=None if prev_sig is None else prev_sig[row_ix],
                curr_sig=None if curr_sig is None else curr_sig[col_ix],
            )
        else:
            raise ValueError(f"Unknown linker_backend: {cfg.linker_backend!r}")
        local_rows, local_cols = linear_sum_assignment(cost)
        for row, col in zip(local_rows, local_cols):
            if cost[row, col] < 1e6:
                assignments.append((rows[int(row)], cols[int(col)]))
    return assignments


def _joint_hypothesis_assignments(
    prev_phys: np.ndarray,
    curr_phys: np.ndarray,
    cfg: Config,
    prev_intensity: Optional[np.ndarray],
    curr_intensity: Optional[np.ndarray],
    prev_velocity: Optional[np.ndarray],
    prev_image_features: Optional[np.ndarray],
    curr_image_features: Optional[np.ndarray],
) -> List[Tuple[int, int]]:
    """Select continuation and 1->2 division hypotheses in one objective.

    A division is one binary optimization variable that claims its parent and
    both daughters together.  This avoids committing to a 1:1 edge before the
    daughter pair has been evaluated.
    """
    learned = cfg.linker_backend in {"learned_joint", "image_learned_joint"}
    gate = cfg.learned_linker_gate_um if learned else cfg.max_link_dist_um
    prev_sig = neighbor_signature(prev_phys, cfg) if cfg.use_rich_linking and not learned else None
    curr_sig = neighbor_signature(curr_phys, cfg) if cfg.use_rich_linking and not learned else None
    selected: List[Tuple[int, int]] = []
    unmatched = float(cfg.joint_hypothesis_unmatched_cost)

    for rows, cols in _candidate_components(prev_phys, curr_phys, gate):
        row_ix = np.asarray(rows, dtype=int)
        col_ix = np.asarray(cols, dtype=int)
        if learned:

            cost, dist = build_learned_link_cost(
                prev_phys[row_ix], curr_phys[col_ix],
                cfg.image_linker_path if cfg.linker_backend == "image_learned_joint" else cfg.learned_linker_path,
                gate_um=gate,
                prev_image_features=None if prev_image_features is None else prev_image_features[row_ix],
                curr_image_features=None if curr_image_features is None else curr_image_features[col_ix],
            )
        else:
            cost, dist = build_link_cost(
                prev_phys[row_ix], curr_phys[col_ix], cfg,
                prev_intensity=None if prev_intensity is None else prev_intensity[row_ix],
                curr_intensity=None if curr_intensity is None else curr_intensity[col_ix],
                prev_velocity=None if prev_velocity is None or len(prev_velocity) != len(prev_phys) else prev_velocity[row_ix],
                prev_sig=None if prev_sig is None else prev_sig[row_ix],
                curr_sig=None if curr_sig is None else curr_sig[col_ix],
            )

        hypotheses: List[Tuple[int, Tuple[int, ...], float]] = []
        for r in range(len(rows)):
            for c in range(len(cols)):
                if cost[r, c] < 1e6:
                    hypotheses.append((r, (c,), 2.0 * unmatched - float(cost[r, c])))
            if not cfg.detect_divisions:
                continue
            nearby = np.flatnonzero(dist[r] <= cfg.div_parent_dist_um)
            for offset, a in enumerate(nearby):
                for b in nearby[offset + 1:]:
                    sister = float(np.linalg.norm(curr_phys[col_ix[a]] - curr_phys[col_ix[b]]))
                    if sister > cfg.div_sister_dist_um:
                        continue
                    midpoint = 0.5 * (curr_phys[col_ix[a]] + curr_phys[col_ix[b]])
                    midpoint_residual = float(np.linalg.norm(prev_phys[row_ix[r]] - midpoint))
                    if cfg.div_use_midpoint_gate and midpoint_residual > cfg.div_midpoint_dist_um:
                        continue
                    combined_displacement = float(dist[r, a] + dist[r, b])
                    asymmetry = abs(float(dist[r, a] - dist[r, b])) / max(combined_displacement, 1e-6)
                    geometry = (
                        midpoint_residual / cfg.div_midpoint_dist_um
                        + sister / cfg.div_sister_dist_um
                        + 0.5 * asymmetry
                    )
                    if prev_velocity is not None and len(prev_velocity) == len(prev_phys):
                        predicted_midpoint = prev_phys[row_ix[r]] + prev_velocity[row_ix[r]]
                        temporal_residual = float(np.linalg.norm(predicted_midpoint - midpoint))
                        geometry += 0.5 * temporal_residual / cfg.div_midpoint_dist_um
                    joint_cost = float(cost[r, a] + cost[r, b]) + cfg.joint_division_geometry_weight * geometry
                    hypotheses.append((r, (a, b), 3.0 * unmatched - joint_cost))

        useful = [hypothesis for hypothesis in hypotheses if hypothesis[2] > 0]
        if not useful:
            continue
        constraint_rows, constraint_cols = [], []
        objective = np.empty(len(useful), dtype=np.float64)
        for h, (r, targets, benefit) in enumerate(useful):
            constraint_rows.append(r)
            constraint_cols.append(h)
            for target in targets:
                constraint_rows.append(len(rows) + target)
                constraint_cols.append(h)
            objective[h] = -benefit + 1e-12 * h
        constraints = csc_matrix(
            (np.ones(len(constraint_rows)), (constraint_rows, constraint_cols)),
            shape=(len(rows) + len(cols), len(useful)),
        )
        result = milp(
            c=objective,
            integrality=np.ones(len(useful), dtype=int),
            bounds=Bounds(0.0, 1.0),
            constraints=LinearConstraint(constraints, 0.0, 1.0),
            options={"presolve": True},
        )
        if not result.success or result.x is None:
            raise RuntimeError(f"Joint linking optimization failed: {result.message}")
        for value, (r, targets, _) in zip(result.x, useful):
            if value > 0.5:
                selected.extend((rows[r], cols[target]) for target in targets)
    return selected


def link_frames(
    prev_ids: Sequence[int],
    prev_xyz: np.ndarray,
    curr_ids: Sequence[int],
    curr_xyz: np.ndarray,
    cfg: Config,
    prev_intensity: Optional[np.ndarray] = None,
    curr_intensity: Optional[np.ndarray] = None,
    prev_velocity: Optional[np.ndarray] = None,
    next_xyz: Optional[np.ndarray] = None,
    prev_image_features: Optional[np.ndarray] = None,
    curr_image_features: Optional[np.ndarray] = None,
) -> List[Tuple[int, int]]:
    """Hungarian assignment with rich costs plus division pass."""
    if len(prev_ids) == 0 or len(curr_ids) == 0:
        return []

    scale = cfg.scale_array
    prev_phys = prev_xyz.astype(np.float64) * scale[None, :]
    curr_phys = curr_xyz.astype(np.float64) * scale[None, :]

    joint_backend = cfg.linker_backend in {"classical_joint", "learned_joint", "image_learned_joint"}
    assignment_fn = _joint_hypothesis_assignments if joint_backend else _component_assignments
    assignment_cfg = cfg
    if (
        joint_backend and cfg.detect_divisions and cfg.div_min_count_gain > 0
        and len(curr_ids) - len(prev_ids) < cfg.div_min_count_gain
    ):
        assignment_cfg = cfg.copy_with(detect_divisions=False)
    assignments = assignment_fn(
        prev_phys, curr_phys, assignment_cfg, prev_intensity, curr_intensity, prev_velocity,
        prev_image_features, curr_image_features,
    )

    edges: List[Tuple[int, int]] = []
    parent_children: Dict[int, List[int]] = defaultdict(list)
    matched_curr = set()

    for r, c in assignments:
        edges.append((int(prev_ids[r]), int(curr_ids[c])))
        parent_children[int(r)].append(int(c))
        matched_curr.add(int(c))

    allow_div = cfg.detect_divisions
    if allow_div and cfg.div_min_count_gain > 0 and len(curr_ids) - len(prev_ids) < cfg.div_min_count_gain:
        allow_div = False

    if allow_div and not joint_backend:
        parent_tree = cKDTree(prev_phys)
        for c in range(len(curr_ids)):
            if c in matched_curr:
                continue
            best_p, best_score = None, np.inf
            candidate_parents = parent_tree.query_ball_point(curr_phys[c], cfg.div_parent_dist_um)
            for p in candidate_parents:
                if len(parent_children.get(p, [])) != 1:
                    continue
                parent_dist = float(np.linalg.norm(prev_phys[p] - curr_phys[c]))
                sister = parent_children[p][0]
                sister_dist = float(np.linalg.norm(curr_phys[c] - curr_phys[sister]))
                if sister_dist > cfg.div_sister_dist_um:
                    continue
                if cfg.div_use_midpoint_gate:
                    midpoint = 0.5 * (curr_phys[c] + curr_phys[sister])
                    mid_dist = float(np.linalg.norm(prev_phys[p] - midpoint))
                    if mid_dist > cfg.div_midpoint_dist_um:
                        continue
                score = float(parent_dist + 0.25 * sister_dist)
                if cfg.div_symmetry_weight > 0:
                    d_parent_sister = float(np.linalg.norm(prev_phys[p] - curr_phys[sister]))
                    d_parent_child = parent_dist
                    if d_parent_sister > 1e-6:
                        asym = abs(d_parent_child - d_parent_sister) / d_parent_sister
                        score += cfg.div_symmetry_weight * asym
                if score < best_score:
                    best_p, best_score = p, score
            if best_p is not None:
                edges.append((int(prev_ids[best_p]), int(curr_ids[c])))
                parent_children[best_p].append(c)
                matched_curr.add(c)

    return edges


def close_frame_gaps(
    frame_ids: List[List[int]],
    frame_centroids: List[np.ndarray],
    frame_intensities: List[np.ndarray],
    existing_edges: List[Tuple[int, int]],
    cfg: Config,
) -> List[Tuple[int, int]]:
    """
    Link detections across one skipped frame (t-2 -> t) for unmatched nodes.

    Recovers short occlusions without full multi-frame optimization.
    """
    if not cfg.gap_close_enabled or len(frame_ids) < 3:
        return []

    scale = cfg.scale_array
    gap_edges: List[Tuple[int, int]] = []
    used_targets: set[int] = set()
    for e in existing_edges:
        used_targets.add(int(e[1]))

    for t in range(2, len(frame_ids)):
        prev2_ids = frame_ids[t - 2]
        prev1_ids = set(frame_ids[t - 1])
        curr_ids = frame_ids[t]
        if not prev2_ids or not curr_ids:
            continue

        linked_from_prev2 = {
            int(s)
            for s, u in existing_edges + gap_edges
            if int(s) in prev2_ids and int(u) in prev1_ids
        }
        linked_to_curr = {
            int(u)
            for s, u in existing_edges + gap_edges
            if int(u) in curr_ids and int(s) in prev1_ids
        }

        unmatched_prev_idx = [i for i, nid in enumerate(prev2_ids) if int(nid) not in linked_from_prev2]
        unmatched_curr_idx = [
            j
            for j, nid in enumerate(curr_ids)
            if int(nid) not in linked_to_curr and int(nid) not in used_targets
        ]
        if not unmatched_prev_idx or not unmatched_curr_idx:
            continue

        prev_xyz = frame_centroids[t - 2][unmatched_prev_idx]
        curr_xyz = frame_centroids[t][unmatched_curr_idx]
        prev_int = frame_intensities[t - 2][unmatched_prev_idx]
        curr_int = frame_intensities[t][unmatched_curr_idx]
        prev_node_ids = [prev2_ids[i] for i in unmatched_prev_idx]
        curr_node_ids = [curr_ids[j] for j in unmatched_curr_idx]

        trial_cfg = cfg.copy_with(max_link_dist_um=cfg.gap_close_dist_um)
        links = link_frames(
            prev_node_ids,
            prev_xyz,
            curr_node_ids,
            curr_xyz,
            trial_cfg,
            prev_intensity=prev_int,
            curr_intensity=curr_int,
            prev_velocity=None,
        )
        for s, u in links:
            gap_edges.append((int(s), int(u)))
            used_targets.add(int(u))

    return gap_edges


def link_sequence_global(
    frame_ids: List[List[int]],
    frame_centroids: List[np.ndarray],
    frame_intensities: List[np.ndarray],
    cfg: Config,
) -> List[Tuple[int, int]]:
    """Optimize continuation and gap links jointly over an entire sequence.

    Each observation has separate source and target copies, allowing at most one
    predecessor and one continuation successor. Forward-only candidate edges make
    the result acyclic. A constrained adjacent-frame pass can add one second child
    to a continuing parent when the configured division geometry is satisfied.
    """
    import networkx as nx

    graph = nx.Graph()
    scale = cfg.scale_array
    max_gap = min(int(cfg.global_max_frame_gap), max(len(frame_ids) - 1, 1))
    for source_t in range(len(frame_ids) - 1):
        source_xyz = frame_centroids[source_t].astype(np.float64) * scale
        if not len(source_xyz):
            continue
        for gap in range(1, max_gap + 1):
            target_t = source_t + gap
            if target_t >= len(frame_ids):
                break
            target_xyz = frame_centroids[target_t].astype(np.float64) * scale
            if not len(target_xyz):
                continue
            gate = cfg.max_link_dist_um * gap
            tree = cKDTree(target_xyz)
            for source_i, neighbors in enumerate(tree.query_ball_point(source_xyz, gate)):
                for target_i in neighbors:
                    dist = float(np.linalg.norm(source_xyz[source_i] - target_xyz[target_i]))
                    cost = dist / gate + cfg.global_gap_penalty * (gap - 1)
                    if len(frame_intensities[source_t]) and len(frame_intensities[target_t]):
                        a = float(frame_intensities[source_t][source_i])
                        b = float(frame_intensities[target_t][target_i])
                        cost += cfg.intensity_lambda * abs(a - b) / max(a, b, 1.0)
                    benefit = 2.0 * cfg.global_unmatched_cost - cost
                    if benefit > 0:
                        graph.add_edge(
                            ("source", int(frame_ids[source_t][source_i])),
                            ("target", int(frame_ids[target_t][target_i])),
                            weight=float(benefit),
                        )
    matching = nx.algorithms.matching.max_weight_matching(graph, maxcardinality=False, weight="weight")
    edges = []
    for left, right in matching:
        if left[0] == "target":
            left, right = right, left
        if left[0] == "source" and right[0] == "target":
            edges.append((int(left[1]), int(right[1])))

    if cfg.detect_divisions:
        id_location = {
            int(node_id): (t, i)
            for t, ids in enumerate(frame_ids)
            for i, node_id in enumerate(ids)
        }
        children: Dict[int, List[int]] = defaultdict(list)
        used_targets = set()
        for source, target in edges:
            children[source].append(target)
            used_targets.add(target)
        for t in range(1, len(frame_ids)):
            if cfg.div_min_count_gain > 0 and len(frame_ids[t]) - len(frame_ids[t - 1]) < cfg.div_min_count_gain:
                continue
            prev_phys = frame_centroids[t - 1].astype(np.float64) * scale
            curr_phys = frame_centroids[t].astype(np.float64) * scale
            if not len(prev_phys) or not len(curr_phys):
                continue
            parent_tree = cKDTree(prev_phys)
            for child_i, child_id in enumerate(frame_ids[t]):
                child_id = int(child_id)
                if child_id in used_targets:
                    continue
                best_parent = None
                best_score = np.inf
                for parent_i in parent_tree.query_ball_point(curr_phys[child_i], cfg.div_parent_dist_um):
                    parent_id = int(frame_ids[t - 1][parent_i])
                    if len(children.get(parent_id, [])) != 1:
                        continue
                    sister_id = children[parent_id][0]
                    sister_t, sister_i = id_location[sister_id]
                    if sister_t != t:
                        continue
                    sister_dist = float(np.linalg.norm(curr_phys[child_i] - curr_phys[sister_i]))
                    if sister_dist > cfg.div_sister_dist_um:
                        continue
                    parent_dist = float(np.linalg.norm(prev_phys[parent_i] - curr_phys[child_i]))
                    if cfg.div_use_midpoint_gate:
                        midpoint = 0.5 * (curr_phys[child_i] + curr_phys[sister_i])
                        if float(np.linalg.norm(prev_phys[parent_i] - midpoint)) > cfg.div_midpoint_dist_um:
                            continue
                    score = parent_dist + 0.25 * sister_dist
                    if cfg.div_symmetry_weight > 0:
                        sister_parent_dist = float(np.linalg.norm(prev_phys[parent_i] - curr_phys[sister_i]))
                        if sister_parent_dist > 1e-6:
                            score += cfg.div_symmetry_weight * abs(parent_dist - sister_parent_dist) / sister_parent_dist
                    if score < best_score:
                        best_parent, best_score = parent_id, score
                if best_parent is not None:
                    edges.append((best_parent, child_id))
                    children[best_parent].append(child_id)
                    used_targets.add(child_id)
    return sorted(edges)


def prune_isolated_nodes(
    node_rows: List[dict],
    edge_rows: List[dict],
    cfg: Optional[Config] = None,
) -> Tuple[List[dict], List[dict], Dict[str, int]]:
    """Annotate isolated detections without deleting raw observations.

    The historical function name is retained for API compatibility.  Nodes that
    would previously have been removed now receive a QC status and reason; edges
    are returned unchanged so graph evidence is never silently discarded.
    """
    incident = set()
    for e in edge_rows:
        incident.add(int(e["source_id"]))
        incident.add(int(e["target_id"]))

    if cfg is not None and cfg.prune_soft_neighbors:
        by_time: Dict[int, List[dict]] = defaultdict(list)
        for r in node_rows:
            by_time[int(r["t"])].append(r)
        scale = cfg.scale_array
        gate = cfg.prune_neighbor_dist_um

        def _nearby(r: dict) -> bool:
            if int(r["node_id"]) in incident:
                return True
            t = int(r["t"])
            pos = np.array([r["z"], r["y"], r["x"]], dtype=np.float64) * scale
            for dt in (-1, 1):
                for other in by_time.get(t + dt, []):
                    op = np.array([other["z"], other["y"], other["x"]], dtype=np.float64) * scale
                    if float(np.linalg.norm(pos - op)) <= gate:
                        return True
            return False

        accepted_ids = {int(r["node_id"]) for r in node_rows if _nearby(r)}
    else:
        accepted_ids = incident

    flagged = 0
    for row in node_rows:
        if int(row["node_id"]) in accepted_ids:
            row["qc_status"] = "pass"
            row["filter_reason"] = ""
        else:
            row["qc_status"] = "flagged"
            row["filter_reason"] = "isolated_no_temporal_edge"
            flagged += 1
    return node_rows, edge_rows, {
        "removed_isolated": 0,
        "flagged_isolated": flagged,
        "kept_nodes": len(node_rows),
        "kept_edges": len(edge_rows),
    }


def annotate_lineage_ids(node_rows: List[dict], edge_rows: List[dict]) -> None:
    """Add observation, track, lineage, and division-parent semantics in place."""
    by_id = {int(row["node_id"]): row for row in node_rows}
    children: Dict[int, List[int]] = defaultdict(list)
    parent: Dict[int, int] = {}
    for edge in edge_rows:
        source = int(edge["source_id"])
        target = int(edge["target_id"])
        children[source].append(target)
        parent[target] = source

    next_track = 1
    next_lineage = 1
    ordered = sorted(node_rows, key=lambda r: (int(r["t"]), int(r["node_id"])))
    for row in ordered:
        node_id = int(row["node_id"])
        source = parent.get(node_id)
        if source is None or source not in by_id:
            row["track_id"] = next_track
            row["lineage_id"] = next_lineage
            row["generation"] = 0
            next_track += 1
            next_lineage += 1
        else:
            source_row = by_id[source]
            row["lineage_id"] = int(source_row["lineage_id"])
            if len(children[source]) >= 2:
                row["track_id"] = next_track
                row["generation"] = int(source_row["generation"]) + 1
                next_track += 1
            else:
                row["track_id"] = int(source_row["track_id"])
                row["generation"] = int(source_row["generation"])
        row["is_division_parent"] = len(children[node_id]) >= 2
        row["division_status"] = "predicted_candidate" if row["is_division_parent"] else "not_applicable"
        row.setdefault("qc_status", "pass")
        row.setdefault("filter_reason", "")


def graph_invariant_errors(
    node_rows: List[dict], edge_rows: List[dict], allowed_gap: int = 1,
) -> List[str]:
    """Return deterministic violations of the lineage graph contract."""
    by_id = {int(row["node_id"]): row for row in node_rows}
    indegree: Counter[int] = Counter()
    outdegree: Counter[int] = Counter()
    adjacency: Dict[int, List[int]] = defaultdict(list)
    errors: List[str] = []
    for edge in edge_rows:
        source = int(edge["source_id"])
        target = int(edge["target_id"])
        if source not in by_id or target not in by_id:
            errors.append(f"dangling edge {source}->{target}")
            continue
        indegree[target] += 1
        outdegree[source] += 1
        adjacency[source].append(target)
        dt = int(by_id[target]["t"]) - int(by_id[source]["t"])
        if dt <= 0:
            errors.append(f"non-forward edge {source}->{target}")
        elif dt > allowed_gap:
            errors.append(f"gap exceeds {allowed_gap} at {source}->{target}")
    errors.extend(f"in-degree >1 at {node}" for node, degree in indegree.items() if degree > 1)
    errors.extend(f"out-degree >2 at {node}" for node, degree in outdegree.items() if degree > 2)
    for source, targets in adjacency.items():
        if len(targets) >= 2:
            source_t = int(by_id[source]["t"])
            target_times = {int(by_id[target]["t"]) for target in targets}
            if target_times != {source_t + 1}:
                errors.append(f"invalid daughter timing at {source}")

    state: Dict[int, int] = {}
    def visit(node: int) -> bool:
        if state.get(node) == 1:
            return True
        if state.get(node) == 2:
            return False
        state[node] = 1
        if any(visit(child) for child in adjacency.get(node, [])):
            return True
        state[node] = 2
        return False
    if any(visit(node) for node in by_id if state.get(node, 0) == 0):
        errors.append("cycle detected")
    return errors


def count_divisions(edge_rows: List[dict]) -> int:
    src_counts = Counter(int(e["source_id"]) for e in edge_rows)
    return sum(1 for c in src_counts.values() if c >= 2)


## Detector


In [ ]:
"""Detector backends."""

from __future__ import annotations

from typing import Optional, Protocol, Tuple

import numpy as np



class CellDetector(Protocol):
    def detect(
        self,
        vol: np.ndarray,
        cfg: Config,
        prev_count: Optional[int] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        ...


class PeakDetector:
    """Classical anisotropy-aware peak detector (Gaussian or DoG band-pass)."""

    def detect(
        self,
        vol: np.ndarray,
        cfg: Config,
        prev_count: Optional[int] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        return detect_cells(vol, cfg, prev_count=prev_count)


class DogPeakDetector(PeakDetector):
    """Peak detector with DoG band-pass preprocessing enabled."""

    def detect(
        self,
        vol: np.ndarray,
        cfg: Config,
        prev_count: Optional[int] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        cfg = cfg.copy_with(use_dog_bandpass=True, detector_backend="peaks_dog")
        return detect_cells(vol, cfg, prev_count=prev_count)


def get_detector(cfg: Config) -> CellDetector:
    if cfg.detector_backend == "learned":

        return TemporalUNetDetector(cfg)
    if cfg.detector_backend == "peaks_dog" or cfg.use_dog_bandpass:
        return DogPeakDetector()
    return PeakDetector()


## Tuning


In [ ]:
"""Systematic hyperparameter search on training data (v1.5)."""

from __future__ import annotations

import time
from pathlib import Path
from collections import defaultdict
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd



def _f1(precision: float, recall: float) -> float:
    denom = precision + recall
    return 2.0 * precision * recall / denom if denom else 0.0


def evaluate_sparse_graph(
    gt_nodes: pd.DataFrame,
    gt_edges: pd.DataFrame,
    pred_edges: Sequence[Tuple[int, int]],
    pred_id_to_gt: Dict[int, int],
    evaluable_times: Optional[Sequence[int]] = None,
) -> Dict[str, float]:
    """Evaluate links and divisions only where sparse GEFF labels support it.

    A predicted edge is evaluable only when both predicted endpoints matched
    annotated GT nodes. Edges with either endpoint unmatched are unknown, not
    false positives. GT recall is restricted to annotated edges whose endpoint
    frames were present in ``gt_nodes``.
    """
    node_time = {
        int(row.node_id): int(row.t)
        for row in gt_nodes[["node_id", "t"]].itertuples(index=False)
    }
    annotated_times = (
        set(node_time.values())
        if evaluable_times is None
        else {int(t) for t in evaluable_times}
    )
    raw_gt_edges = [] if gt_edges is None else [
        (int(s), int(t))
        for s, t in gt_edges[["source_id", "target_id"]].to_numpy()
    ]
    gt_set = {
        (s, t)
        for s, t in raw_gt_edges
        if s in node_time and t in node_time
        and node_time[s] in annotated_times and node_time[t] in annotated_times
    }

    evaluable_pairs: List[Tuple[int, int]] = []
    unknown_edges = 0
    for s, t in pred_edges:
        gs, gt = pred_id_to_gt.get(s), pred_id_to_gt.get(t)
        if gs is None or gt is None or gs not in node_time or gt not in node_time:
            unknown_edges += 1
            continue
        evaluable_pairs.append((int(gs), int(gt)))

    predicted_pair_set = set(evaluable_pairs)
    edge_tp = sum(pair in gt_set for pair in evaluable_pairs)
    edge_fp = len(evaluable_pairs) - edge_tp
    edge_fn = len(gt_set - predicted_pair_set)
    edge_precision = edge_tp / (edge_tp + edge_fp) if edge_tp + edge_fp else 0.0
    edge_recall = edge_tp / (edge_tp + edge_fn) if edge_tp + edge_fn else 0.0

    gt_children: Dict[int, set[int]] = defaultdict(set)
    for s, t in gt_set:
        gt_children[s].add(t)
    gt_division_parents = {s for s, children in gt_children.items() if len(children) >= 2}

    pred_children: Dict[int, List[int]] = defaultdict(list)
    for s, t in pred_edges:
        pred_children[int(s)].append(int(t))
    raw_divisions = {s: children for s, children in pred_children.items() if len(children) >= 2}
    division_tp = 0
    division_fp = 0
    unknown_divisions = 0
    confirmed_gt_parents: set[int] = set()
    for source, children in raw_divisions.items():
        gt_source = pred_id_to_gt.get(source)
        gt_child_ids = [pred_id_to_gt.get(child) for child in children]
        if gt_source is None or any(child is None for child in gt_child_ids):
            unknown_divisions += 1
            continue
        matched_children = {int(child) for child in gt_child_ids if child is not None}
        if (
            int(gt_source) in gt_division_parents
            and len(matched_children & gt_children[int(gt_source)]) >= 2
        ):
            division_tp += 1
            confirmed_gt_parents.add(int(gt_source))
        else:
            division_fp += 1
    division_fn = len(gt_division_parents - confirmed_gt_parents)
    division_precision = (
        division_tp / (division_tp + division_fp)
        if division_tp + division_fp else 0.0
    )
    division_recall = (
        division_tp / (division_tp + division_fn)
        if division_tp + division_fn else 0.0
    )

    return {
        "edge_precision": float(edge_precision),
        "edge_recall": float(edge_recall),
        "edge_f1": float(_f1(edge_precision, edge_recall)),
        "edge_tp": float(edge_tp),
        "edge_fp": float(edge_fp),
        "edge_fn": float(edge_fn),
        "edge_unknown": float(unknown_edges),
        "edge_pred_evaluable": float(len(evaluable_pairs)),
        "edge_pred_unknown": float(unknown_edges),
        "edge_gt_evaluable": float(len(gt_set)),
        "division_precision": float(division_precision),
        "division_recall": float(division_recall),
        "division_f1": float(_f1(division_precision, division_recall)),
        "division_tp": float(division_tp),
        "division_fp": float(division_fp),
        "division_fn": float(division_fn),
        "division_unknown": float(unknown_divisions),
        "divisions": float(len(raw_divisions)),
    }


def _match_nodes_to_gt(
    pred_xyz: np.ndarray,
    gt_frame: pd.DataFrame,
    scale: np.ndarray,
    gate_um: float = MATCH_GATE_UM,
) -> Dict[int, int]:
    """Map predicted index -> GT node ID with optimal one-to-one matching."""
    if len(pred_xyz) == 0 or len(gt_frame) == 0:
        return {}
    gt_xyz = gt_frame[["z", "y", "x"]].to_numpy(dtype=np.float64)
    gt_ids = gt_frame["node_id"].astype(int).to_numpy()
    pred_phys = pred_xyz.astype(np.float64) * scale
    gt_phys = gt_xyz * scale
    from scipy.optimize import linear_sum_assignment

    dist = np.sqrt(((pred_phys[:, None, :] - gt_phys[None, :, :]) ** 2).sum(axis=2))
    row_idx, col_idx = linear_sum_assignment(np.where(dist <= gate_um, dist, 1e6))
    mapping: Dict[int, int] = {}
    for i, j in zip(row_idx, col_idx):
        if dist[i, j] <= gate_um:
            mapping[int(i)] = int(gt_ids[j])
    return mapping


def score_config_on_sample(
    train_dir: Path,
    name: str,
    cfg: Config,
    n_frames: int = 4,
) -> Dict[str, float]:
    """Proxy score using production linking and sparse-label-safe metrics."""
    geff_path = find_geff_path(train_dir, name)
    if geff_path is None:
        return {
            "label_available": 0.0,
            "recall": np.nan,
            "edge_proxy": np.nan,
            "divisions": np.nan,
            "score": np.nan,
        }
    gt_nodes, gt_edges = load_geff_graph(geff_path)
    if gt_nodes is None:
        return {
            "label_available": 0.0,
            "recall": np.nan,
            "edge_proxy": np.nan,
            "divisions": np.nan,
            "score": np.nan,
        }

    zarr_path = train_dir / f"{name}.zarr"
    shape, dtype = read_zarr_meta(zarr_path)
    voxel_size, frame_interval = read_physical_metadata(zarr_path)
    if voxel_size is not None or frame_interval is not None:
        cfg = cfg.copy_with(
            voxel_size_zyx_um=voxel_size or cfg.voxel_size_zyx_um,
            frame_interval_s=frame_interval or cfg.frame_interval_s,
        )
    scale = cfg.scale_array

    recalls = []
    pred_edges: List[Tuple[int, int]] = []
    prev_xyz = np.empty((0, 3))
    prev_int = np.empty((0,))
    prev_image_features = None
    prev_vel = None
    prev_ids: List[int] = []
    pred_id = 1
    pred_id_to_gt: Dict[int, int] = {}

    labeled_ts = gt_nodes["t"].to_numpy()
    start_t = int(labeled_ts.min()) if len(labeled_ts) else 0
    start_t = min(start_t, max(shape[0] - n_frames, 0))
    frame_range = range(start_t, min(start_t + n_frames, shape[0]))

    for loop_i, t in enumerate(frame_range):
        vol = load_volume(zarr_path, t, shape, dtype)
        prev_count = len(prev_xyz) if loop_i else None
        coords, _, intens = cached_detection(
            zarr_path,
            t,
            cfg,
            prev_count,
            lambda: detect_cells(vol, cfg, prev_count=prev_count),
        )
        if cfg.linker_backend == "image_learned":
            curr_image_features = extract_local_image_features(vol, coords)
        else:
            curr_image_features = None
        gt_t = gt_nodes[gt_nodes["t"] == t]
        # Hungarian (one-to-one) assignment FIRST; one prediction can never
        # satisfy two GT cells.
        frame_map = _match_nodes_to_gt(coords, gt_t, scale) if len(coords) and len(gt_t) else {}
        if len(gt_t):
            recalls.append(len(frame_map) / len(gt_t))
        curr_ids = []
        for i in range(len(coords)):
            pid = pred_id
            pred_id += 1
            curr_ids.append(pid)
            if i in frame_map:
                pred_id_to_gt[pid] = frame_map[i]

        if t > 0 and len(prev_ids) and len(curr_ids):
            links = link_frames(
                prev_ids,
                prev_xyz,
                curr_ids,
                coords,
                cfg,
                prev_intensity=prev_int,
                curr_intensity=intens,
                prev_velocity=prev_vel,
                prev_image_features=prev_image_features,
                curr_image_features=curr_image_features,
            )
            pred_edges.extend((int(s), int(u)) for s, u in links)

            if len(links):
                vel = {}
                pos = {pid: prev_xyz[i] for i, pid in enumerate(prev_ids)}
                pos.update({pid: coords[i] for i, pid in enumerate(curr_ids)})
                for s, u in links:
                    if s in pos and u in pos:
                        vel[u] = (pos[u] - pos[s]) * scale
                prev_vel = np.asarray(
                    [vel.get(pid, np.zeros(3)) for pid in curr_ids],
                    dtype=np.float64,
                )

        prev_xyz, prev_int, prev_ids = coords, intens, curr_ids
        prev_image_features = curr_image_features

    recall = float(np.mean(recalls)) if recalls else 0.0
    graph_metrics = evaluate_sparse_graph(
        gt_nodes,
        gt_edges,
        pred_edges,
        pred_id_to_gt,
        evaluable_times=list(frame_range),
    )
    score = (
        0.45 * recall
        + 0.45 * graph_metrics["edge_f1"]
        + 0.10 * graph_metrics["division_f1"]
    )
    return {
        "label_available": 1.0,
        "recall": recall,
        "edge_proxy": graph_metrics["edge_f1"],
        "score": score,
        **graph_metrics,
    }


def hyperparameter_search(
    train_dir: Path,
    cfg: Config,
    sample_limit: int = 3,
    frames: int = 4,
) -> Tuple[Config, pd.DataFrame]:
    """
    Grid search key hyperparameters on training samples.

    Returns best config and a results table.
    """
    names = list_datasets(train_dir)[:sample_limit]
    if not names:
        return cfg, pd.DataFrame()

    grid = {
        "thresh_rel": [0.26, 0.28, 0.30, 0.32],
        "max_link_dist_um": [10.0, 11.0, 12.0],
        "div_parent_dist_um": [11.0, 12.0],
        "div_sister_dist_um": [7.0, 7.5, 8.0],
        "nms_radius_um": [2.5, 2.65],
    }

    rows: List[dict] = []
    best_cfg = cfg
    best_score = -1.0

    base = {
        "thresh_rel": cfg.thresh_rel,
        "max_link_dist_um": cfg.max_link_dist_um,
        "div_parent_dist_um": cfg.div_parent_dist_um,
        "div_sister_dist_um": cfg.div_sister_dist_um,
        "nms_radius_um": cfg.nms_radius_um,
    }

    def _iter_grid(d, keys, i=0, cur=None):
        cur = cur or {}
        if i == len(keys):
            yield cur
            return
        k = keys[i]
        for v in d[k]:
            cur[k] = v
            yield from _iter_grid(d, keys, i + 1, cur)

    for params in _iter_grid(grid, list(grid.keys())):
        t0 = time.time()
        trial = cfg.copy_with(**params)
        scores = [score_config_on_sample(train_dir, n, trial, frames) for n in names]
        runtime_s = time.time() - t0
        score_values = [s["score"] for s in scores]
        mean_score = float(np.nanmean(score_values)) if np.isfinite(score_values).any() else np.nan
        row = {
            **params,
            "mean_score": mean_score,
            "runtime_s": round(runtime_s, 2),
            "n_samples": len(names),
            "valid_scores": int(np.isfinite(score_values).sum()),
        }
        row.update({f"{k}_{i}": s[k] for i, s in enumerate(scores) for k in ("recall", "edge_proxy")})
        rows.append(row)
        if np.isfinite(mean_score) and mean_score > best_score:
            best_score = mean_score
            best_cfg = trial.copy_with(**params)

    results = pd.DataFrame(rows).sort_values("mean_score", ascending=False)
    return best_cfg, results


def single_knob_candidates() -> List[Dict[str, float]]:
    """
    Low-risk one-parameter candidates for post-v4 leaderboard optimization.

    These avoid v1.5 regression flags (gap close, soft prune, division symmetry)
    and keep each trial attributable to exactly one parameter.
    """
    return [
        {},
        {"max_link_dist_um": 9.5},
        {"max_link_dist_um": 10.0},
        {"max_link_dist_um": 10.5},
        {"max_link_dist_um": 11.5},
        {"thresh_rel": 0.26},
        {"thresh_rel": 0.28},
        {"thresh_rel": 0.32},
        {"thresh_rel": 0.34},
        {"nms_radius_um": 2.50},
        {"nms_radius_um": 2.80},
        {"div_parent_dist_um": 11.0},
        {"div_parent_dist_um": 13.0},
        {"div_sister_dist_um": 7.0},
        {"div_sister_dist_um": 8.0},
    ]


def _candidate_name(params: Dict[str, float]) -> Tuple[str, str, float | None]:
    if not params:
        return "baseline", "baseline", None
    if len(params) != 1:
        raise ValueError(f"Expected one-parameter candidate, got {params}")
    knob, value = next(iter(params.items()))
    return f"{knob}={value}", knob, float(value)


def bootstrap_ci(
    values: Sequence[float],
    n_boot: int = 2000,
    ci: float = 0.95,
    random_state: int = 42,
) -> Tuple[float, float, float]:
    """Return mean and bootstrap CI after dropping non-finite values."""
    arr = np.asarray(values, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if not len(arr):
        return np.nan, np.nan, np.nan
    mean = float(arr.mean())
    if len(arr) == 1:
        return mean, mean, mean
    rng = np.random.default_rng(random_state)
    samples = rng.choice(arr, size=(n_boot, len(arr)), replace=True).mean(axis=1)
    alpha = (1.0 - ci) / 2.0
    return mean, float(np.quantile(samples, alpha)), float(np.quantile(samples, 1.0 - alpha))


def paired_bootstrap_delta(
    baseline: Sequence[float],
    candidate: Sequence[float],
    n_boot: int = 2000,
    ci: float = 0.95,
    random_state: int = 42,
) -> Tuple[float, float, float]:
    """Bootstrap a paired candidate-minus-baseline delta by dataset."""
    base = np.asarray(baseline, dtype=np.float64)
    cand = np.asarray(candidate, dtype=np.float64)
    if base.shape != cand.shape:
        raise ValueError("baseline and candidate must contain aligned per-dataset scores")
    valid = np.isfinite(base) & np.isfinite(cand)
    delta = cand[valid] - base[valid]
    return bootstrap_ci(delta, n_boot=n_boot, ci=ci, random_state=random_state)


def single_knob_sweep(
    train_dir: Path,
    cfg: Config,
    sample_limit: int = 5,
    frames: int = 6,
    candidates: Optional[Iterable[Dict[str, float]]] = None,
) -> Tuple[Config, pd.DataFrame]:
    """
    Score low-risk one-knob candidates on sparse train labels.

    The public leaderboard is still the final judge, but this avoids blind
    manual trials by ranking candidates with the same proxy used for local
    hyperparameter search.
    """
    names = list_datasets(train_dir)[:sample_limit]
    if not names:
        return cfg, pd.DataFrame()

    rows: List[dict] = []
    candidate_scores: Dict[str, np.ndarray] = {}
    best_cfg = cfg
    best_score = -1.0
    candidate_list = list(candidates) if candidates is not None else single_knob_candidates()

    for params in candidate_list:
        candidate, knob, value = _candidate_name(params)
        trial = cfg.copy_with(**params)
        # Explicit threshold trials must not be overwritten by later density calibration.
        if knob == "thresh_rel":
            trial = trial.copy_with(run_density_calibration=False)

        t0 = time.time()
        scores = [score_config_on_sample(train_dir, n, trial, frames) for n in names]
        runtime_s = time.time() - t0
        score_values = np.array([s["score"] for s in scores], dtype=np.float64)
        candidate_scores[candidate] = score_values
        recall_values = np.array([s["recall"] for s in scores], dtype=np.float64)
        edge_values = np.array([s["edge_proxy"] for s in scores], dtype=np.float64)
        division_values = np.array([s["divisions"] for s in scores], dtype=np.float64)
        label_values = np.array([s["label_available"] for s in scores], dtype=np.float64)
        valid_scores = int(np.isfinite(score_values).sum())
        mean_score = float(np.nanmean(score_values)) if valid_scores else np.nan
        mean_recall = float(np.nanmean(recall_values)) if np.isfinite(recall_values).any() else np.nan
        mean_edge = float(np.nanmean(edge_values)) if np.isfinite(edge_values).any() else np.nan
        mean_divisions = float(np.nanmean(division_values)) if np.isfinite(division_values).any() else np.nan
        row = {
            "candidate": candidate,
            "knob": knob,
            "value": value,
            "mean_score": mean_score,
            "mean_recall": mean_recall,
            "mean_edge_proxy": mean_edge,
            "mean_divisions": mean_divisions,
            "runtime_s": round(runtime_s, 2),
            "n_samples": len(names),
            "valid_scores": valid_scores,
            "labels_found": int(label_values.sum()),
            "frames": frames,
            "run_density_calibration": trial.run_density_calibration,
        }
        for k in (
            "thresh_rel",
            "max_link_dist_um",
            "nms_radius_um",
            "div_parent_dist_um",
            "div_sister_dist_um",
        ):
            row[k] = getattr(trial, k)
        rows.append(row)
        if np.isfinite(mean_score) and mean_score > best_score:
            best_score = mean_score
            best_cfg = trial

    results = pd.DataFrame(rows)
    baseline_rows = results[results["candidate"] == "baseline"]
    if len(baseline_rows) and np.isfinite(float(baseline_rows.iloc[0]["mean_score"])):
        baseline_score = float(baseline_rows.iloc[0]["mean_score"])
        deltas = {
            name: paired_bootstrap_delta(candidate_scores["baseline"], values)
            for name, values in candidate_scores.items()
        }
        results["delta_vs_baseline"] = results["candidate"].map(lambda name: deltas[name][0])
        results["delta_ci_low"] = results["candidate"].map(lambda name: deltas[name][1])
        results["delta_ci_high"] = results["candidate"].map(lambda name: deltas[name][2])
        results["significant_improvement"] = results["delta_ci_low"] > 0.0
    else:
        results["delta_vs_baseline"] = np.nan
        results["delta_ci_low"] = np.nan
        results["delta_ci_high"] = np.nan
        results["significant_improvement"] = False
    results = results.sort_values("mean_score", ascending=False, na_position="last")
    return best_cfg, results


## Batch export


In [ ]:
"""Batch pipeline for exporting lineage CSV across multiple test volumes."""

from __future__ import annotations

import gc
import logging
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


logger = logging.getLogger(__name__)


def _node_row(dataset: str, node_id: int, t: int, zyx, raw_detection_score=None, intensity=None) -> dict:
    z, y, x = [float(v) for v in zyx]
    return {
        "dataset": dataset,
        "row_type": "node",
        "node_id": int(node_id),
        "t": int(t),
        "z": z,
        "y": y,
        "x": x,
        "source_id": -1,
        "target_id": -1,
        "raw_detection_score": float(raw_detection_score) if raw_detection_score is not None else np.nan,
        "intensity": float(intensity) if intensity is not None else np.nan,
    }


def _edge_row(dataset: str, source_id: int, target_id: int) -> dict:
    return {
        "dataset": dataset,
        "row_type": "edge",
        "node_id": -1,
        "t": -1,
        "z": -1,
        "y": -1,
        "x": -1,
        "source_id": int(source_id),
        "target_id": int(target_id),
    }


def calibrate_detection(
    train_dir: Path,
    cfg: Config,
    sample_limit: int = 4,
    frames_per_sample: int = 5,
) -> float:
    """Sweep thresh_rel on training samples to match per-frame density."""
    detector = get_detector(cfg)
    names = list_datasets(train_dir)[:sample_limit]
    if not names:
        return cfg.thresh_rel

    grid = [0.26, 0.30, 0.34, 0.38]
    best_thresh, best_err = cfg.thresh_rel, np.inf

    for thresh in grid:
        cfg.thresh_rel = thresh
        ratios = []
        for name in names:
            est = read_estimated_nodes(train_dir / f"{name}.geff")
            if est is None or est <= 0:
                continue
            zarr_path = train_dir / f"{name}.zarr"
            shape, dtype = read_zarr_meta(zarr_path)
            n_frames = min(frames_per_sample, shape[0])
            n_pred = 0
            for t in range(n_frames):
                vol = load_volume(zarr_path, t, shape, dtype)
                coords, _, _ = detector.detect(vol, cfg)
                n_pred += len(coords)
            avg_per_frame = n_pred / max(n_frames, 1)
            target_per_frame = est / max(shape[0], 1)
            ratios.append(avg_per_frame / max(target_per_frame, 1e-6))
        if ratios:
            ratio = float(np.median(ratios))
            err = (ratio - 1.0) ** 2
            if ratio > 1.0:
                err += 0.75 * (ratio - 1.0) ** 2
            if err < best_err:
                best_err, best_thresh = err, thresh

    cfg.thresh_rel = best_thresh
    return best_thresh


def apply_train_tuning(cfg: Config) -> Tuple[Config, Optional[pd.DataFrame]]:
    """Run v1.5 hyperparameter search when enabled and train data exists."""
    if not cfg.run_hyperparameter_search or cfg.train_dir is None:
        return cfg, None
    logger.info("Running hyperparameter search (v%s)...", PIPELINE_VERSION)
    best, table = hyperparameter_search(
        cfg.train_dir,
        cfg,
        sample_limit=cfg.hyperparam_sample_limit,
        frames=cfg.hyperparam_frames,
    )
    if len(table):
        out_path = Path(cfg.hyperparam_results_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        table.to_csv(out_path, index=False)
        logger.info("Wrote hyperparameter results to %s\n%s", out_path, table.head(5).to_string(index=False))
    return best, table


def process_dataset(
    split_dir: Path,
    name: str,
    cfg: Config,
) -> Tuple[List[dict], Dict[str, object]]:
    """Run detection + tracking on one dataset."""
    zarr_path = split_dir / f"{name}.zarr"
    shape, dtype = read_zarr_meta(zarr_path)
    voxel_size, frame_interval = read_physical_metadata(zarr_path)
    if voxel_size is not None or frame_interval is not None:
        cfg = cfg.copy_with(
            voxel_size_zyx_um=voxel_size or cfg.voxel_size_zyx_um,
            frame_interval_s=frame_interval or cfg.frame_interval_s,
        )
    detector = get_detector(cfg)
    n_t = shape[0]

    node_rows: List[dict] = []
    edge_rows: List[dict] = []
    frame_ids: List[List[int]] = []
    frame_centroids: List[np.ndarray] = []
    frame_intensities: List[np.ndarray] = []
    frame_image_features: List[np.ndarray] = []
    node_id = 1
    prev_count: Optional[int] = None
    frame_counts: List[int] = []
    velocity_by_id: Dict[int, np.ndarray] = {}
    position_by_id: Dict[int, np.ndarray] = {}

    for t in range(n_t):
        vol = load_volume(zarr_path, t, shape, dtype)
        coords, scores, intens = detector.detect(vol, cfg, prev_count=prev_count)
        if cfg.linker_backend == "image_learned":
            image_features = extract_local_image_features(vol, coords)
        else:
            image_features = np.empty((len(coords), 0), dtype=np.float32)
        del vol
        gc.collect()

        if len(coords):
            order = np.lexsort((coords[:, 2], coords[:, 1], coords[:, 0]))
            coords = coords[order]
            scores = scores[order]
            intens = intens[order]
            image_features = image_features[order]

        curr_ids = list(range(node_id, node_id + len(coords)))
        node_id += len(coords)
        frame_ids.append(curr_ids)
        frame_centroids.append(coords)
        frame_intensities.append(intens)
        frame_image_features.append(image_features)
        frame_counts.append(len(coords))
        prev_count = len(coords)

        for nid, pt, raw_score, intensity in zip(curr_ids, coords, scores, intens):
            node_rows.append(_node_row(name, nid, t, pt, raw_score, intensity))
            position_by_id[int(nid)] = pt.astype(np.float64)

    if cfg.tracking_backend == "global":
        for s, u in link_sequence_global(frame_ids, frame_centroids, frame_intensities, cfg):
            edge_rows.append(_edge_row(name, s, u))
    for t in range(1, n_t) if cfg.tracking_backend == "framewise" else ():
        prev_ids = frame_ids[t - 1]
        curr_ids = frame_ids[t]
        prev_xyz = frame_centroids[t - 1]
        curr_xyz = frame_centroids[t]
        prev_int = frame_intensities[t - 1]
        curr_int = frame_intensities[t]
        prev_vel = np.asarray(
            [velocity_by_id.get(pid, np.zeros(3, dtype=np.float64)) for pid in prev_ids],
            dtype=np.float64,
        )

        links = link_frames(
            prev_ids,
            prev_xyz,
            curr_ids,
            curr_xyz,
            cfg,
            prev_intensity=prev_int,
            curr_intensity=curr_int,
            prev_velocity=prev_vel if cfg.use_rich_linking else None,
            prev_image_features=frame_image_features[t - 1] if cfg.linker_backend == "image_learned" else None,
            curr_image_features=frame_image_features[t] if cfg.linker_backend == "image_learned" else None,
        )
        for s, u in links:
            edge_rows.append(_edge_row(name, s, u))
            if s in position_by_id and u in position_by_id:
                velocity_by_id[u] = (
                    position_by_id[u] - position_by_id[s]
                ) * cfg.scale_array

    pair_edges = [(int(e["source_id"]), int(e["target_id"])) for e in edge_rows]
    for s, u in close_frame_gaps(frame_ids, frame_centroids, frame_intensities, pair_edges, cfg) if cfg.tracking_backend == "framewise" else ():
        edge_rows.append(_edge_row(name, s, u))

    nodes_before = len(node_rows)
    edges_before = len(edge_rows)
    div_before = count_divisions(edge_rows)

    if cfg.prune_isolated_nodes:
        node_rows, edge_rows, prune_stats = prune_isolated_nodes(node_rows, edge_rows, cfg)
    else:
        prune_stats = {"removed_isolated": 0, "flagged_isolated": 0}

    annotate_lineage_ids(node_rows, edge_rows)
    allowed_gap = cfg.global_max_frame_gap if cfg.tracking_backend == "global" else (2 if cfg.gap_close_enabled else 1)
    invariant_errors = graph_invariant_errors(node_rows, edge_rows, allowed_gap=allowed_gap)
    if invariant_errors:
        raise ValueError("Invalid tracking graph: " + "; ".join(invariant_errors))

    stats = {
        "dataset": name,
        "shape": shape,
        "nodes_before_prune": nodes_before,
        "edges_before_prune": edges_before,
        "nodes_after_prune": len(node_rows),
        "edges_after_prune": len(edge_rows),
        "divisions": count_divisions(edge_rows),
        "divisions_before_prune": div_before,
        "removed_isolated": prune_stats.get("removed_isolated", 0),
        "flagged_isolated": prune_stats.get("flagged_isolated", 0),
        "count_min": int(min(frame_counts)) if frame_counts else 0,
        "count_max": int(max(frame_counts)) if frame_counts else 0,
        "count_mean": float(np.mean(frame_counts)) if frame_counts else 0.0,
    }
    return node_rows + edge_rows, stats


def build_submission(cfg: Optional[Config] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Process all test datasets and write a combined lineage CSV."""
    cfg = cfg or Config()
    if cfg.use_competition_preset:
        cfg = cfg.competition_v4_preset()
    cfg.resolve_paths()
    if cfg.test_dir is None:
        raise FileNotFoundError("Test directory not found")

    if cfg.train_dir is not None:
        cfg, _ = apply_train_tuning(cfg)
        if cfg.run_density_calibration:
            calibrate_detection(cfg.train_dir, cfg, cfg.eda_sample_limit, cfg.calibration_frames)
        else:
            logger.info("Skipping density calibration; using thresh_rel=%.2f", cfg.thresh_rel)

    datasets = list_datasets(cfg.test_dir)
    all_rows: List[dict] = []
    all_stats = []
    t0 = time.time()

    logger.info(
        f"Pipeline v{PIPELINE_VERSION} | detector={cfg.detector_backend} | "
        f"dog={cfg.use_dog_bandpass} | thresh_rel={cfg.thresh_rel:.2f}"
    )

    for i, name in enumerate(datasets, 1):
        rows, stats = process_dataset(cfg.test_dir, name, cfg)
        all_rows.extend(rows)
        all_stats.append(stats)
        elapsed = time.time() - t0
        logger.info(
            f"[{i}/{len(datasets)}] {name}: "
            f"{stats['nodes_after_prune']} nodes, {stats['edges_after_prune']} edges, "
            f"{stats['divisions']} divisions ({elapsed:.0f}s)"
        )

    cols = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    sub = pd.DataFrame(all_rows)[cols]
    node_mask = sub["row_type"].eq("node")
    sub.loc[node_mask, ["z", "y", "x"]] = np.rint(
        sub.loc[node_mask, ["z", "y", "x"]].astype(float)
    ).astype(int)
    sub.index.name = "id"
    sub.to_csv(cfg.output_path)
    logger.info("Wrote %s (%s rows) in %.1f min", cfg.output_path, f"{len(sub):,}", (time.time()-t0)/60)
    return sub, pd.DataFrame(all_stats)


def validate_submission(sub: pd.DataFrame, test_dir: Path) -> None:
    """Assert lineage CSV format and graph consistency."""
    expected = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    assert list(sub.columns) == expected, f"Wrong columns: {list(sub.columns)}"
    assert len(sub) > 0, "Empty submission"

    datasets = set(list_datasets(test_dir))
    assert datasets.issubset(set(sub["dataset"].unique())), "Missing test datasets"

    nodes = sub[sub.row_type == "node"]
    edges = sub[sub.row_type == "edge"]
    assert sub["row_type"].isin(["node", "edge"]).all(), "Invalid row_type"
    assert sub["dataset"].isin(datasets).all(), "Unexpected dataset"
    assert not nodes.duplicated(["dataset", "node_id"]).any(), "Duplicate node_id within dataset"
    assert (nodes[["node_id", "t", "z", "y", "x"]] >= 0).all().all()
    assert (nodes[["source_id", "target_id"]] == -1).all().all()
    assert (edges[["node_id", "t", "z", "y", "x"]] == -1).all().all()
    assert (edges[["source_id", "target_id"]] >= 0).all().all()

    for ds, grp in sub.groupby("dataset"):
        nset = set(grp.loc[grp.row_type == "node", "node_id"].astype(int))
        e = grp[grp.row_type == "edge"]
        assert e["source_id"].astype(int).isin(nset).all(), f"dangling source in {ds}"
        assert e["target_id"].astype(int).isin(nset).all(), f"dangling target in {ds}"
        times = grp.loc[grp.row_type == "node"].set_index("node_id")["t"]
        if len(e):
            source_times = e["source_id"].map(times).to_numpy()
            target_times = e["target_id"].map(times).to_numpy()
            assert (source_times < target_times).all(), f"non-forward edge in {ds}"
            assert not e.duplicated(["source_id", "target_id"]).any(), f"duplicate edge in {ds}"
            assert e.groupby("target_id").size().max() <= 1, f"multiple parents in {ds}"
            assert e.groupby("source_id").size().max() <= 2, f"more than two daughters in {ds}"

    logger.info("Validation passed")


def local_recall_proxy(
    gt_nodes: pd.DataFrame,
    pred_xyz: np.ndarray,
    t: int,
    scale: np.ndarray = np.array(SCALE),
    gate_um: float = MATCH_GATE_UM,
) -> float:
    """Fraction of GT nodes at time t matched by a prediction within gate_um."""
    gt_t = gt_nodes[gt_nodes["t"] == t]
    if len(gt_t) == 0 or len(pred_xyz) == 0:
        return np.nan
    gt_phys = gt_t[["z", "y", "x"]].to_numpy(dtype=np.float64) * scale
    pred_phys = pred_xyz.astype(np.float64) * scale
    matched = 0
    for g in gt_phys:
        d = np.sqrt(((pred_phys - g) ** 2).sum(axis=1))
        if np.min(d) <= gate_um:
            matched += 1
    return matched / len(gt_t)


## 1. Configuration

| Mode | Setting | Expected score |
|------|---------|----------------|
| **V7 baseline** | `USE_V10_DOG = False` | 0.659 (validated) |
| **V10 DoG** | `USE_V10_DOG = True` | Detection experiment |

V10 swaps only the **detection front-end** (DoG band-pass). Tracking and division stay on the v4 preset.


In [ ]:
CFG = Config()
CFG.resolve_paths()

USE_V10_DOG = True  # False = V7 baseline (0.659); True = V10 DoG experiment

if USE_V10_DOG:
    CFG = CFG.competition_v10_dog_preset()
    print("V10 DoG preset — detector=peaks_dog")
else:
    CFG = CFG.competition_v4_preset()
    print("V7 baseline preset")

print(f"Pipeline v{PIPELINE_VERSION}")
print(f"detector: {CFG.detector_backend} | dog={CFG.use_dog_bandpass}")
print(f"train: {CFG.train_dir}")
print(f"test:  {CFG.test_dir}")
print(f"out:   {CFG.output_path}")


## 2. Candidate sweep (V7 only)

Skip automatically for V10 DoG. For V7, ranks one-knob candidates when train labels are readable.


In [ ]:
RUN_SINGLE_KNOB_SWEEP = True
AUTO_APPLY_SWEEP_WINNER = True
AUTO_APPLY_BEST_NON_BASELINE_IF_BASELINE_WINS = False

if CFG.use_dog_bandpass:
    print("V10 DoG active — skipping V8 knob sweep (not applicable to DoG front-end)")
elif RUN_SINGLE_KNOB_SWEEP and CFG.train_dir is not None:
    CFG, sweep_table = single_knob_sweep(
        CFG.train_dir,
        CFG,
        sample_limit=5,
        frames=6,
    )
    sweep_path = Path("/kaggle/working/single_knob_sweep.csv") if Path("/kaggle/working").is_dir() else Path("single_knob_sweep.csv")
    sweep_table.to_csv(sweep_path, index=False)
    display(sweep_table.head(12))
    has_proxy_scores = sweep_table["mean_score"].notna().any()
    if not has_proxy_scores:
        print(
            "No usable train-label proxy scores were produced. "
            "GEFF labels were not found/readable for these train samples."
        )
        print("Keeping V7 baseline. Choose a manual one-knob override only if you want a leaderboard experiment.")
        CFG = Config().competition_v4_preset()
        CFG.resolve_paths()
    else:
        top_candidate = str(sweep_table.iloc[0]["candidate"])
        non_baseline = sweep_table[sweep_table["candidate"] != "baseline"]
        best_non_baseline = non_baseline.iloc[0] if len(non_baseline) else None
        print(
            "Best proxy candidate:",
            top_candidate,
            f"thresh_rel={CFG.thresh_rel:.3f}",
            f"max_link_dist_um={CFG.max_link_dist_um:.1f}",
            f"nms={CFG.nms_radius_um:.2f}",
            f"div_parent={CFG.div_parent_dist_um:.1f}",
            f"div_sister={CFG.div_sister_dist_um:.1f}",
            f"density_calibration={CFG.run_density_calibration}",
        )
        if best_non_baseline is not None:
            print(
                "Best non-baseline candidate:",
                best_non_baseline["candidate"],
                f"score={best_non_baseline['mean_score']:.4f}",
                f"delta_vs_baseline={best_non_baseline['delta_vs_baseline']:.4f}",
            )
        if top_candidate == "baseline":
            print("Baseline won the proxy sweep. This will replay V7 unless you choose a non-baseline candidate.")
            if AUTO_APPLY_BEST_NON_BASELINE_IF_BASELINE_WINS and best_non_baseline is not None:
                knob = str(best_non_baseline["knob"])
                CFG = Config().competition_v4_preset()
                CFG.resolve_paths()
                CFG = CFG.copy_with(**{knob: float(best_non_baseline["value"])})
                if knob == "thresh_rel":
                    CFG = CFG.copy_with(run_density_calibration=False)
                print("Applied best non-baseline candidate:", best_non_baseline["candidate"])
    print(f"Wrote {sweep_path}")
    if not AUTO_APPLY_SWEEP_WINNER:
        CFG = Config().competition_v4_preset()
        CFG.resolve_paths()
        print("AUTO_APPLY_SWEEP_WINNER=False — reset to V7 baseline")
elif CFG.train_dir is None:
    print("No train directory — skipping candidate sweep")
else:
    print("Skipping candidate sweep (RUN_SINGLE_KNOB_SWEEP=False)")


## 3. V10 DoG overrides (optional)

Tune DoG sigmas if the default V10 run under/over-detects. **One override at a time.**

| Knob | Typical effect |
|------|----------------|
| `dog_sigma_small_um` | Inner scale — smaller = finer blobs |
| `dog_sigma_large_um` | Outer scale — larger = stronger background removal |


In [ ]:
V10_DOG_OVERRIDES = {
    # "dog_sigma_small_um": 0.8,
    # "dog_sigma_large_um": 3.2,
}

if V10_DOG_OVERRIDES:
    if len(V10_DOG_OVERRIDES) != 1:
        raise ValueError(f"Use exactly one DoG override: {V10_DOG_OVERRIDES}")
    CFG = CFG.copy_with(**V10_DOG_OVERRIDES)
    print("V10 DoG overrides:", V10_DOG_OVERRIDES)


## 4. Manual V7 override (optional, V7 only)

Use this only if you want to override the sweep winner.  
**Rule:** keep at most **one** key in `V8_OVERRIDES`. Threshold overrides automatically disable density calibration so they are not overwritten.

| Knob | Typical effect |
|------|----------------|
| `max_link_dist_um` | Edge recall vs false links |
| `thresh_rel` | Node count / precision |
| `nms_radius_um` | Duplicate peak suppression |
| `div_parent_dist_um` / `div_sister_dist_um` | Division metric |


In [ ]:
# === Optional manual V8 override: keep at most ONE key ===
V8_OVERRIDES = {
    # "max_link_dist_um": 10.0,
    # "max_link_dist_um": 10.5,
    # "thresh_rel": 0.28,
    # "thresh_rel": 0.32,
    # "nms_radius_um": 2.5,
    # "div_parent_dist_um": 11.0,
    # "div_sister_dist_um": 7.0,
    # "div_sister_dist_um": 8.0,
}

if V8_OVERRIDES:
    if len(V8_OVERRIDES) != 1:
        raise ValueError(f"Use exactly one V8 override, got {len(V8_OVERRIDES)}: {V8_OVERRIDES}")
    CFG = CFG.copy_with(**V8_OVERRIDES)
    if "thresh_rel" in V8_OVERRIDES:
        CFG = CFG.copy_with(run_density_calibration=False)
        print("Disabled density calibration so explicit thresh_rel is preserved")
    print("V8 overrides applied:", V8_OVERRIDES)
else:
    print("No manual overrides — using sweep winner if enabled, otherwise V7 baseline")


## 5. Build submission

Runs density calibration when enabled, then processes all test volumes.


In [ ]:
submission, stats_df = build_submission(CFG)
print(submission["row_type"].value_counts())
display(submission.head(8))
display(stats_df)
submission.to_csv(CFG.output_path)
if CFG.test_dir is not None:
    validate_submission(submission, CFG.test_dir)
print(f"Wrote {CFG.output_path} ({len(submission):,} rows)")
